In [5]:
!pip install -q --ignore-installed earthengine-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.6/250.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# -*- coding: utf-8 -*-
"""NATIONAL_FOREST_THRESHOLD_VM0047_ELIGIBILITY_ANALYSIS.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1votE0K2KhjTM1Izk_gPkJqGZUn6_PBpQ
"""

try:
    import ee  # noqa: F401
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'earthengine-api'])

"""
================================================================================
NATIONAL FOREST THRESHOLD / VM0047 ELIGIBILITY ANALYSIS -- v7 (FIXED)
Dry-season-only baseline persistence, area-based patch contiguity,
gated carve-out fallback

ADDED IN v10.3 (export / manual-verification fix):
  - Outputs go to a new time-stamped folder per run (VM0047_outputs_<RUN_ID>),
    and the zip is named with the RUN_ID. An old CSV can no longer be
    mistaken for this run's result (the old fixed file names were why the
    CSV showed "all Pass" while the panel showed 8 classifier fails).
  - Per-parcel CSVs are written right after Section 9 (checkpoint), before
    the slow Section 9c validation, and then re-read and reconciled against
    the in-memory results. A MISMATCH warning is printed if they differ.
  - Section 9c is wrapped in try/except so a timeout can't stop the exports.
  - NEW Manual_Verification_Register (CSV + first Excel sheet): every plot
    the classifier failed, showing REVIEWED (override applied) or PENDING
    (still needs the multi-date Google Earth Pro check), plus
    processing-error and recency-flagged plots.
  - Every row carries Run_ID and Console_Line (the exact processing-panel line).
  - Plots with a processing error are labelled 'Error - Not Processed' and
    go to the Flagged file. Before, they dropped out of both files.

ADDED IN v10 (Saleem's Section 8a diagnostic follow-up):
  - STUDY_SUBDISTRICTS fixed to ['KUDAL', 'SAWANTWADI'] (all caps) to match
    the actual study_area shapefile attribute casing -- this was silently
    causing Section 9c's independent ESA WorldCover / Hansen GFC benchmark
    to skip both subdistricts entirely. Also added a _norm_sub() helper and
    normalized NAME_TO_SUBDIST so the Python-side (pandas) comparisons in
    Sections 9b/14b are casing-robust going forward, though the GEE-side
    ee.Filter.eq in Section 9c still needs STUDY_SUBDISTRICTS to match the
    asset's raw attribute values verbatim.
  - PERSISTENCE_FRACTION lowered 0.70 -> 0.30 and AMPLITUDE_LOW_THRESHOLD
    raised 0.15 -> 0.38, per Saleem's diagnostic showing the Konkan
    monsoon climate's cloud gaps/dry-season leaf fall and wet-dry NDVI
    contrast were pushing tree_mask to zero everywhere. IMPORTANT CAVEAT:
    Saleem's quoted percentiles were computed over the WHOLE study area
    (scrub + crop + trees together), which isn't the right population to
    calibrate a tree/scrub discriminator against, since scrub and crop are
    expected to score high on exactly these two measures. Section 8a now
    ALSO reports both statistics restricted to height>=2m pixels only --
    check that conditional line, not the whole-area line, before trusting
    these two values, and re-run Section 9c after any further change to
    confirm the pipeline's forest % still lines up with ESA WorldCover /
    Hansen GFC (a new explicit pass/fail gate on this was added right
    after Section 9c's per-subdistrict verdict).

ADDED IN v9 (PRR 12 response -- Stratum 3 / boundary):
  - NEW Section 2c: -15 m inward negative buffer applied to ALL 127
    plots (not a selected subset), reprojected to EPSG:32643 (UTM
    Zone 43N) before buffering so the -15 m is a true metric distance.
    Applied uniformly regardless of a plot's eventual pass/fail result
    -- this is a fixed edge-effect correction (removes road/boundary
    mixed pixels), not a result-dependent trim. Plots too small/narrow
    to survive a -15 m inward buffer (this WILL happen for slivers)
    fall back to a small fixed-radius disc around the centroid, and
    every fallback is logged to console AND written as its own column
    (Buffer_Fallback_Used) in the parcel CSV so a VVB reviewer can see
    exactly which plots' effective analysis footprint isn't the full
    -15 m inset. The buffered boundary is exported as its own
    shapefile (Section 2c/13, NOT gated behind RUN_FALLBACK_CARVEOUT --
    that flag only controls sub-parcel forest carve-out, a separate
    and unrelated operation).
  - All zonal statistics in Section 9 now run on the buffered geometry
    (post -15 m inset) rather than the raw digitized parcel boundary.
    Total_Parcel_Area_ha (and every Area_*/Pct_* column) is computed on
    that buffered footprint -- it's what was actually sampled. A new
    Raw_Parcel_Area_ha column carries the original digitized parcel
    area for reference, and Buffer_Fallback_Used (True/False) flags any
    plot that fell back to the centroid disc, so both are auditable
    side by side in the same CSV row.
  - Per-plot 1 m classified raster + 30 m evidence raster exports
    (Section 14) are RE-ENABLED (previously commented out). Subdistrict
    exports (14b) are unchanged and still run alongside them.
  - NO threshold constants were changed in this version: NDVI_VEG_THRESHOLD,
    NDVI_TREE_THRESHOLD, MIN_CONTIGUOUS_COVER_PCT, and PRIMARY_PATCH_AREA_HA
    are exactly as calibrated in v8 against India's DNA definition. A PRR
    request to treat a much higher NDVI value (~0.69) as still non-forest
    was NOT implemented here because it isn't derivable from anything in
    this pipeline -- if that number comes from an independent regression
    against ground-truth plots, it should be added as a documented,
    reproducible calibration step (with the full ground-truth sample it
    was fit on), not hardcoded as a pass/fail cutoff.
  - Isolated/scattered single trees before June 2021 are already handled
    correctly by the existing patch-area sieve: any connected tree patch
    below PRIMARY_PATCH_AREA_HA (0.05 ha) is excluded from the
    "qualifying" contiguous-cover calculation (Section 9), so scattered
    stems cannot by themselves push a parcel over the 15% threshold.
    Section 9c's independent ESA WorldCover / Hansen GFC benchmark check
    (kept from v8) is the right place to confirm the subdistrict overall
    still shows real forest patches even while individual plots don't --
    that comparison is now also written per-plot-adjacent in Section 9d
    below for the flagged parcels specifically.

ADDED IN v8:
  - PRIMARY_PATCH_AREA_HA lowered 0.10 -> 0.05 ha to match India's DNA
    minimum mappable area exactly (crown cover >=15%, height >=2 m,
    patch >=0.05 ha). One constant, propagates everywhere.
  - NEW Section 9c: subdistrict-wide forest validation. Runs the same
    tree classification over the FULL Kudal/Sawantwadi admin polygons
    and compares against ESA WorldCover 2021, Hansen GFC (end-2020),
    and a manually-entered FSI ISFR figure. Prints a verdict per
    subdistrict + writes VM0047_v9_Subdistrict_Forest_Validation.csv.
    This is the independent check Section 9b could never provide.

ADDED IN THIS VERSION (on top of the v7 fixes):
  9b. SUBDISTRICT-LEVEL RESULTS AGGREGATION
      - subdist_summary_df: per-subdistrict rollup (parcel counts, pass/fail
        counts, total area, class-area totals, mean contiguous tree cover %)
      - CSV export + new "Subdistrict_Summary" sheet in the Excel workbook
  14b. SUBDISTRICT-LEVEL RASTER EXPORTS
      - Dissolves all plot geometries sharing the same SUB_DIST value into
        one subdistrict boundary per entry in STUDY_SUBDISTRICTS
      - Exports the same two stacks as the per-plot loop (1m classified
        stack, 30m evidence stack), clipped to each subdistrict boundary
        instead of each parcel
      - The "qualifying tree patch" band for the subdistrict export is the
        UNION of every parcel's qualifying (>= PRIMARY_PATCH_AREA_HA) patches
        across all parcels in that subdistrict
      - These tasks are appended to _all_tasks so they get picked up by the
        same poll_tasks() call as the per-plot exports

--------------------------------------------------------------------------------
ALL OUTPUTS RESTORED (per user request):
  - CSV: All_Parcels_Detail, Fully_Passing_Parcels, Flagged_Parcels,
    Carved_Eligible_Subpolygons (if fallback run), Eligibility_Summary,
    and Subdistrict_Summary.
  - Excel: VM0047_v9_Eligibility_Report.xlsx (all sheets incl. Subdistrict_Summary).
  - Rasters: per-subdistrict 1m classified stack + 30m evidence stack GeoTIFFs
    to Drive folder SUBDISTRICT_EXPORT_FOLDER (Section 14b). Per-plot raster
    exports (Section 14) remain commented out, same as your original script.

  IMPORTANT: the CSV/Excel files above are written to the LOCAL WORKING
  DIRECTORY of wherever you run this script (printed at runtime, Section 11)
  -- NOT to Google Drive. Only the GEE raster exports go to Drive, and only
  after poll_tasks() reports them COMPLETED. If you were looking for the
  CSVs/Excel in Drive, that's why they seemed to be missing.
================================================================================
"""

import ee
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime

# ------------------------------------------------------------------
# v10.3 -- RUN-STAMPED OUTPUT FOLDER (fixes "CSV shows all Pass while the
# processing panel shows Fail - Forest")
#
# ROOT CAUSE: every run wrote to the SAME fixed file names in the working
# directory. If a run stopped before Section 11 (e.g. Section 9c's
# admin-polygon-wide reduceRegion timing out -- note the httplib2 timeout
# warnings), the CSV/Excel left in the folder were from an OLDER run, so the
# files you opened did not match what the panel just printed.
#
# FIX: (a) every run writes into its own time-stamped folder, so an old
# file can never be mistaken for this run's result; (b) parcel results are
# written IMMEDIATELY after Section 9 (checkpoint), before any slow step;
# (c) every CSV row carries the Run_ID and the exact console line printed
# for that plot; (d) after writing, the files are re-read and reconciled
# against the in-memory results -- any mismatch is printed loudly.
# ------------------------------------------------------------------
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = os.path.abspath(f'VM0047_outputs_{RUN_ID}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

def out_path(filename):
    """Absolute path for an output file inside this run's own folder."""
    return os.path.join(OUTPUT_DIR, filename)

print(f'[Run] RUN_ID = {RUN_ID}')
print(f'[Run] All CSV/Excel outputs for THIS run go to: {OUTPUT_DIR}')

# ------------------------------------------------------------------
# 0. AUTHENTICATION / INITIALIZATION
# ------------------------------------------------------------------
PROJECT_ID = 'ee-saleemkhan'

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# ------------------------------------------------------------------
# 1. USER SETTINGS
# ------------------------------------------------------------------
STUDY_AREA_ASSET = 'projects/ee-saleemkhan/assets/EarthMAA_StudyArea'
PLOTS_ASSET      = 'projects/ee-saleemkhan/assets/Re127ProjectPlots'
SUBDIST_FIELD    = 'Sub_dist'
NAME_FIELD       = 'Name'

def _norm_sub(value):
    """Case/whitespace-normalized subdistrict name -- keeps Python-side
    comparisons (df filtering, dict lookups) robust to casing regardless
    of what's actually recorded in the source shapefile attribute table."""
    return str(value).strip().upper() if value is not None else value

STUDY_SUBDISTRICTS = ['KUDAL', 'SAWANTWADI']   # v10: fixed to match the ACTUAL casing
# in the study_area shapefile attribute table ('KUDAL'/'SAWANTWADI', all caps), not
# the title-case guess used in v9. This exact string is also what Section 9c passes
# into ee.Filter.eq(SUBDIST_FIELD, ...) against the raw GEE asset, so it MUST match
# the asset's real attribute values verbatim -- if the asset casing ever changes again,
# update this list to match it exactly (the _norm_sub() calls below only make the
# Python-side (pandas) comparisons casing-robust; they can't fix a server-side
# ee.Filter.eq against an asset whose attribute values don't match this string).

PROJECT_START_YEAR = 2021
PROJECT_START_MONTH = 6

HLS_YEARS = list(range(2013, 2022))   # 2013-2021, HLS only -- no bridge

SEASON_WINDOWS = {
    'winter_dry':      ('01-01', '02-15'),
    'pre_monsoon_dry':  ('03-01', '04-30'),
    'monsoon_wet':      ('07-01', '09-15'),
    'post_monsoon':     ('10-15', '11-30'),
}
SEASONS_PRE_START_2021 = ['winter_dry', 'pre_monsoon_dry']

DRY_SEASONS = ['winter_dry', 'pre_monsoon_dry']

NDVI_VEG_THRESHOLD = 0.35     # general vegetation presence -- drives Herbaceous class only
NDVI_TREE_THRESHOLD = 0.40    # dry-season-only tree-canopy persistence threshold
                               # (documented acceptable range: 0.40-0.42)
PERSISTENCE_FRACTION = 0.30   # v10: lowered from 0.70 per Saleem's Section 8a diagnostic
                               # (Konkan monsoon climate -> heavy cloud gaps + dry-season leaf
                               # fall depress even genuine-forest persistence scores). PROVISIONAL:
                               # Section 8a now also reports this statistic conditioned on
                               # height>=2m pixels specifically (not just the whole-area figure
                               # Saleem quoted) -- re-check that conditional number before trusting
                               # 0.30 long-term, and re-run Section 9c after any change here.
MIN_OBS_FOR_PERSISTENCE = 5   # NOTE: at PERSISTENCE_FRACTION=0.30 this floor matters more than
                               # before -- 2/6 cloud-free dry-season obs already clears 0.30,
                               # so a pixel with very few obs can pass on thin evidence. Consider
                               # raising this if dry_n_obs is low across most of the study area
                               # (see Section 8a diagnostic printout).

MAX_CLOUD_COVER_PCT = 10

HEIGHT_SCRUB_MIN_M = 0.5
MIN_CANOPY_HEIGHT_M = 2.0
AMPLITUDE_LOW_THRESHOLD = 0.55   # v10: raised from 0.15 per Saleem's Section 8a diagnostic.
                                  # PROVISIONAL, same caveat as PERSISTENCE_FRACTION above: the
                                  # 0.47 mean / 0.55 p90 Saleem quoted is the WHOLE-STUDY-AREA
                                  # amplitude (scrub + crop + trees all mixed in), which is not
                                  # the right population to calibrate a tree/scrub discriminator
                                  # against -- scrub and crop are EXPECTED to have high amplitude
                                  # (that's the discriminator's whole logic). What matters is the
                                  # amplitude distribution specifically among height>=2m pixels,
                                  # which Section 8a now reports separately. If that conditional
                                  # p90 is still well above 0.38, this value needs revisiting.
HERB_MIN_PERSISTENCE = 0.30

MIN_CONTIGUOUS_COVER_PCT = 15.0
PURE_FALLOW_MAX_PCT      = 2.0

PRIMARY_PATCH_AREA_HA = 0.05   # v8: India DNA minimum mappable area (was 0.10 in v7).
                               # DNA forest definition: >=15% crown cover, >=2 m height,
                               # >=0.05 ha contiguous patch. Decision logic everywhere
                               # (per-parcel sieve, flagging, carve-out, raster band)
                               # reads this one constant, so nothing else changes.
FOREST_PATCH_AREA_THRESHOLDS_HA = [0.05, 0.10, 0.25, 0.50]   # sensitivity sweep, real hectares
CARVE_MIN_REMAINDER_HA = 0.05   # below this, a carved-out remainder is dropped as noise

HEIGHT_SCALE_M = 1
HLS_SCALE_M = 30
CARVE_VECTORIZE_SCALE_M = 10

# -- v9: boundary inset (PRR 12) --
PLOT_UTM_CRS = 'EPSG:32643'          # WGS 84 / UTM Zone 43N, per PRR 12
BOUNDARY_INSET_M = -15                # negative = inward buffer
CENTROID_FALLBACK_RADIUS_M = 2.0      # small disc used only when a plot
                                       # is too small/narrow to survive
                                       # a -15 m inward buffer
BOUNDARY_EXPORT_FOLDER = 'VM0047_v9_Buffered_Boundary'

TREND_FLAT_BAND = 0.01
GROUP_STABLE = 'Stable'
GROUP_DECLINING = 'Declining'
GROUP_INCREASING = 'Increasing'

EVIDENCE_NEEDED = {
    GROUP_STABLE: 'Verify vegetation is scrub, crops, grass, or other non-tree '
                   'cover rather than forest (see class-area breakdown; high-res '
                   'imagery / field visit for confirmation).',
    GROUP_DECLINING: 'Provide evidence the decline reflects agriculture, drought, '
                      'fire, or pest activity rather than pre-project deforestation.',
    GROUP_INCREASING: 'Provide evidence the increase reflects weed/scrub '
                       'encroachment rather than natural forest regeneration.',
}

LC_CLASSES = {0: 'Bare_Fallow', 1: 'Herbaceous_Grass_Crop', 2: 'Scrub_Shrub', 3: 'Tree_Cover'}

RUN_FALLBACK_CARVEOUT = False

EXPORT_DRIVE_FOLDER = 'VM0047_PerPlot_Rasters_v9'
CARVEOUT_EXPORT_FOLDER = 'VM0047_Revised_Boundary_v9'
SUBDISTRICT_EXPORT_FOLDER = 'VM0047_Subdistrict_Rasters_v9'   # NEW
EXPORT_BUFFER_M = 5

# ------------------------------------------------------------------
# 1c. MANUAL / DESK-REVIEW OVERRIDES -- NEW v10.2
#
# The pixel classifier is retained as the primary, reproducible evidence
# for every plot. This registry exists for the opposite situation: a
# human reviewer has already looked at something the classifier cannot
# see well -- e.g. multi-date Google Earth Pro history spanning several
# WET and DRY seasons since project start, confirming a plot never
# regrew, which rules out the laterite-scrub seasonal-browning false
# positive a single dry-season snapshot can't rule out on its own.
#
# THIS NEVER SILENTLY REPLACES THE ALGORITHMIC RESULT. Every entry here
# produces a SEPARATE Manual_Override_* set of columns alongside an
# untouched Final_Result_Algorithmic column in every CSV -- a VVB (or
# you, in six months) can see the machine result and the human
# correction side by side, with the evidence and reviewer named, rather
# than one silently overwriting the other. Final_Result (the column
# everything downstream buckets on) reflects the reviewed outcome.
#
# status must be exactly one of: 'Confirmed_Non_Forest', 'Confirmed_Forest'
# Add plots here only after a documented multi-date review -- a single
# image is not enough grounds for an override (see the recency-check
# diagnostic in Section 9 for a systematic, non-manual first pass over
# all 127 plots).
# ------------------------------------------------------------------
MANUAL_VERIFICATION_OVERRIDES = {
    '305_Shivram': {
        'status': 'Confirmed_Non_Forest',
        'evidence': 'Multi-date Google Earth Pro historical imagery reviewed from 2021 '
                    'to present, across multiple dry AND wet seasons -- vegetation '
                    'cleared and confirmed NOT to have regrown at any point in that '
                    'window (rules out seasonal laterite-scrub browning as the cause '
                    'of the classifier\'s bare/low-NDVI reading).',
        'verified_by': 'M. Saleem Khan',
        'verified_date': '2026-09-09',
    },
    '303_Manohar_1': {
        'status': 'Confirmed_Non_Forest',
        'evidence': 'Multi-date Google Earth Pro historical imagery reviewed from 2021 '
                    'to present; same basis as 305_Shivram above.',
        'verified_by': 'M. Saleem Khan',
        'verified_date': '2026-09-09',
    },
    '303__Manohar_4': {
        'status': 'Confirmed_Non_Forest',
        'evidence': 'Multi-date Google Earth Pro historical imagery reviewed from 2021 '
                    'to present; same basis as 305_Shivram above.',
        'verified_by': 'M. Saleem Khan',
        'verified_date': '2026-09-09',
    },
    '315_Praveen Desai_1': {
        'status': 'Confirmed_Non_Forest',
        'evidence': 'Multi-date Google Earth Pro historical imagery reviewed from 2021 '
                    'to present; same basis as 305_Shivram above.',
        'verified_by': 'M. Saleem Khan',
        'verified_date': '2026-09-09',
    },
    # ---- v10.3: plots the classifier FAILED that have NOT been reviewed yet.
    # They stay 'Fail - Forest' and appear as "PENDING" in the
    # Manual_Verification_Register export until you complete the same
    # multi-date Google Earth Pro review and uncomment/fill an entry, e.g.:
    #
    # '310_Sunil Vasudev': {
    #     'status': 'Confirmed_Non_Forest',   # or 'Confirmed_Forest'
    #     'evidence': '<what you checked, which dates/seasons, what you saw>',
    #     'verified_by': '<name>',
    #     'verified_date': 'YYYY-MM-DD',
    # },
    # Same template for: '310_Sunil_Vasudev_1_', '312_Rajesh Laomaji',
    # '313_Shivram_Bamboo'
}
# Double-check every key above against the exact 'Name' field spelling in
# Re127ProjectPlots (case- and underscore-sensitive, e.g. "303_Manohar_1" vs
# "303__Manohar_4" have a different number of underscores) -- a typo here
# means the override silently never matches and the algorithmic result is
# used instead, with no warning. Sec. 9 below prints one line per plot that
# an override actually matched, and a summary count at the end, so you can
# confirm all four applied.

# ------------------------------------------------------------------
# 1d. RECENCY-CHECK PARAMETERS -- NEW v10.2
#
# The 2013-2021 persistence bands (Section 6) pool ALL baseline years
# together, so a plot forested for 7-8 years and then cleared just before
# the June 2021 project start can still show high overall persistence --
# exactly the failure mode the 4 manually-reviewed plots above surfaced.
# Rather than relying only on ad hoc manual review, Section 9 also
# computes a RECENT-ONLY dry-season tree-persistence fraction (years
# below) for every one of the 127 plots and flags a large drop between
# the historical and recent signal as "worth a manual look", the same
# kind of check you already did by hand for the 4 plots above. This is
# a DIAGNOSTIC/evidence column only -- it does not change Final_Result
# on its own, so it can't be gamed by itself the way a pass/fail
# threshold could.
# ------------------------------------------------------------------
RECENT_YEARS_FOR_CLEARING_CHECK = [2020, 2021]   # last full dry seasons before project start
RECENCY_DROP_THRESHOLD = 0.40     # historical_frac - recent_frac >= this ...
RECENCY_ABS_LOW_THRESHOLD = 0.30  # ... AND recent_frac itself <= this -> flagged

# ------------------------------------------------------------------
# 1b. EXPORT TASK TRACKING
# ------------------------------------------------------------------
_all_tasks = []  # (task_object, description) tuples

def start_and_track(task, description):
    task.start()
    _all_tasks.append((task, description))
    return task

def safe_band_suffix(value):
    """0.1 -> '0p1', 0.25 -> '0p25' -- GEE band names can't contain '.'"""
    return str(value).replace('.', 'p')

def safe_name(s):
    return ''.join(ch if ch.isalnum() else '_' for ch in str(s))[:80]

def poll_tasks(tasks, poll_interval_sec=30, max_wait_sec=None):
    start_time = time.time()
    pending = {t.id: (t, desc) for t, desc in tasks}
    terminal = {}

    while pending:
        if max_wait_sec and (time.time() - start_time) > max_wait_sec:
            print(f'\n[Monitor] Stopped polling after {max_wait_sec}s -- '
                  f'{len(pending)} task(s) still not in a terminal state.')
            break

        for task_id in list(pending.keys()):
            task, desc = pending[task_id]
            status = task.status()
            state = status.get('state')
            if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
                terminal[task_id] = (desc, state, status.get('error_message'))
                pending.pop(task_id)

        print(f'[Monitor] {len(terminal)} done / {len(terminal) + len(pending)} total '
              f'({len(pending)} still queued or running)...', end='\r')
        if pending:
            time.sleep(poll_interval_sec)

    print('\n\n=== EXPORT TASK RESULTS ===')
    n_ok = sum(1 for _, s, _ in terminal.values() if s == 'COMPLETED')
    n_fail = sum(1 for _, s, _ in terminal.values() if s == 'FAILED')
    n_cancel = sum(1 for _, s, _ in terminal.values() if s == 'CANCELLED')
    print(f'Completed: {n_ok}  |  Failed: {n_fail}  |  Cancelled: {n_cancel}  |  '
          f'Still pending: {len(pending)}')
    if n_fail:
        print('\n--- FAILED TASK DETAILS ---')
        for desc, state, err in terminal.values():
            if state == 'FAILED':
                print(f'  {desc}: {err}')
    return terminal

def check_recent_tasks(n=50):
    ops = ee.data.listOperations()
    for op in ops[:n]:
        meta = op.get('metadata', {})
        desc = meta.get('description', '(no description)')
        state = meta.get('state')
        err = op.get('error', {}).get('message') if 'error' in op else None
        line = f'{state:12s}  {desc}'
        if err:
            line += f'   <-- ERROR: {err}'
        print(line)

# ------------------------------------------------------------------
# 2. LOAD VECTORS
# ------------------------------------------------------------------
study_area = ee.FeatureCollection(STUDY_AREA_ASSET)
plots = ee.FeatureCollection(PLOTS_ASSET)

# --- Diagnostic: confirm NAME_FIELD exists on plots, and SUBDIST_FIELD
#     exists on study_area. Turns out these two attributes live on
#     DIFFERENT assets:
#       plots (Re127ProjectPlots): Name, OID_, Shape_Area, Shape_Leng, system:index
#       study_area (EarthMAA_StudyArea): Dist_LGD, District, OBJECTID, REMARKS,
#         STATE_LGD, STATE_UT, Shape_Area, Shape_Leng, Sub_dist, Subdis_LGD,
#         Subdis_Typ, system:index
#     -- study_area is an admin-boundary layer (LGD district/subdistrict
#     polygons), not a per-plot attribute. Section 2b below spatially
#     joins each plot to whichever study_area polygon it falls inside.
_plots_field_names = plots.first().propertyNames().getInfo()
_study_area_field_names = study_area.first().propertyNames().getInfo()
print(f'\n[Field check] Properties found on plots ({PLOTS_ASSET}):')
print(f'  {sorted(_plots_field_names)}')
print(f'[Field check] Properties found on study_area ({STUDY_AREA_ASSET}):')
print(f'  {sorted(_study_area_field_names)}')
if NAME_FIELD not in _plots_field_names:
    print(f'  [WARNING] NAME_FIELD = "{NAME_FIELD}" is NOT one of the plot properties above. '
          f'Update NAME_FIELD in Section 1 to the exact (case-sensitive) name from the list.')
if SUBDIST_FIELD not in _study_area_field_names:
    print(f'  [WARNING] SUBDIST_FIELD = "{SUBDIST_FIELD}" is NOT one of the study_area properties '
          f'above -- the spatial join in Section 2b will produce empty subdistrict values. Update '
          f'SUBDIST_FIELD in Section 1 to the exact (case-sensitive) name from the study_area list.')

# ------------------------------------------------------------------
# 2b. SPATIAL JOIN: ASSIGN SUBDISTRICT TO EACH PLOT
#
# plots has no subdistrict attribute of its own. study_area does
# (SUBDIST_FIELD), as an admin-boundary polygon layer. For each plot we
# take its centroid and find which study_area polygon contains it, then
# carry that polygon's SUBDIST_FIELD value back onto the plot by Name.
# Centroid-based matching is used (rather than testing the full plot
# polygon) so a plot that straddles two study_area polygons still gets
# exactly one unambiguous subdistrict assignment.
# ------------------------------------------------------------------
print('\nSpatially joining plots to study_area to assign subdistrict ...')

plot_centroids = plots.map(
    lambda f: ee.Feature(f.geometry().centroid(maxError=1), {NAME_FIELD: f.get(NAME_FIELD)})
)

_spatial_filter = ee.Filter.intersects(leftField='.geo', rightField='.geo', maxError=1)
_joined = ee.Join.saveFirst('subdist_match').apply(
    primary=plot_centroids, secondary=study_area, condition=_spatial_filter
)

def _extract_name_subdist(f):
    match = ee.Feature(f.get('subdist_match'))
    return ee.Feature(None, {
        NAME_FIELD: f.get(NAME_FIELD),
        SUBDIST_FIELD: match.get(SUBDIST_FIELD),
    })

_name_subdist_features = ee.FeatureCollection(_joined).map(_extract_name_subdist).getInfo()['features']
NAME_TO_SUBDIST = {
    feat['properties'].get(NAME_FIELD): _norm_sub(feat['properties'].get(SUBDIST_FIELD))
    for feat in _name_subdist_features
}
_n_matched = sum(1 for v in NAME_TO_SUBDIST.values() if v is not None)
print(f'  Matched {_n_matched}/{len(NAME_TO_SUBDIST)} plots to a study_area subdistrict polygon.')
if _n_matched < len(NAME_TO_SUBDIST):
    _unmatched = [k for k, v in NAME_TO_SUBDIST.items() if v is None]
    print(f'  [WARNING] {len(_unmatched)} plot(s) did not intersect any study_area polygon '
          f'(check CRS/coverage): {_unmatched[:10]}{" ..." if len(_unmatched) > 10 else ""}')

# ------------------------------------------------------------------
# 2c. -15 M BOUNDARY INSET, UTM ZONE 43N (EPSG:32643), CENTROID FALLBACK
#     -- NEW v9, PRR 12 response --
#
# Applied to ALL 127 plots, unconditionally, before any classification
# happens. This is a fixed geometric operation (edge-effect / mixed-pixel
# correction), not something that depends on what the classifier later
# finds -- the same -15 m inset is applied whether a plot will pass or
# flag. Reprojecting to EPSG:32643 first means -15 means -15 metres,
# not -15 of whatever unit the source asset happens to be in.
#
# A negative buffer on a small or narrow (sliver) polygon can return an
# empty or invalid geometry -- there's no interior 15 m from every edge
# left. Rather than silently dropping those plots from the analysis,
# each one falls back to a small fixed-radius disc around its centroid,
# and is flagged so the fallback is visible in the audit trail rather
# than hidden inside an otherwise-uniform buffer column.
# ------------------------------------------------------------------
buffered_geom_by_name = {}
buffer_fallback_by_name = {}
buffer_export_features = []

_plot_list_2c = plots.toList(plots.size())
_n_plots_2c = plots.size().getInfo()

print(f'\nApplying {BOUNDARY_INSET_M} m inward buffer in {PLOT_UTM_CRS} to all {_n_plots_2c} plots ...')

for i in range(_n_plots_2c):
    feature = ee.Feature(_plot_list_2c.get(i))
    name_val = feature.get(NAME_FIELD).getInfo()
    geom_utm = feature.geometry().transform(PLOT_UTM_CRS, 1)

    raw_area_m2 = geom_utm.area(maxError=1).getInfo()
    buffered = geom_utm.buffer(BOUNDARY_INSET_M, maxError=1)
    buffered_area_m2 = buffered.area(maxError=1).getInfo()

    used_fallback = buffered_area_m2 <= 0
    if used_fallback:
        buffered = geom_utm.centroid(maxError=1).buffer(CENTROID_FALLBACK_RADIUS_M, maxError=1)
        buffered_area_m2 = buffered.area(maxError=1).getInfo()
        print(f'  [{i+1}/{_n_plots_2c}] {name_val}: -{abs(BOUNDARY_INSET_M)} m inset collapsed the '
              f'polygon (raw area {raw_area_m2/10000.0:.3f} ha) -- FELL BACK to a '
              f'{CENTROID_FALLBACK_RADIUS_M} m centroid disc ({buffered_area_m2/10000.0:.4f} ha). '
              f'Flag this plot for manual boundary review.')

    buffered_wgs84 = buffered.transform('EPSG:4326', 1)
    buffered_geom_by_name[name_val] = buffered_wgs84
    buffer_fallback_by_name[name_val] = used_fallback
    buffer_export_features.append(ee.Feature(buffered_wgs84, {
        NAME_FIELD: name_val,
        'Raw_Area_ha': raw_area_m2 / 10000.0,
        'Buffered_Area_ha': buffered_area_m2 / 10000.0,
        'Buffer_Inset_m': BOUNDARY_INSET_M,
        'Centroid_Fallback': bool(used_fallback),
    }))

n_fallback = sum(buffer_fallback_by_name.values())
print(f'  Done: {_n_plots_2c - n_fallback}/{_n_plots_2c} plots kept the full {BOUNDARY_INSET_M} m inset; '
      f'{n_fallback} used the centroid fallback (see console lines above / '
      f'Centroid_Fallback column in the exported boundary shapefile).')

buffered_boundary_fc = ee.FeatureCollection(buffer_export_features)
t_buffer_boundary = ee.batch.Export.table.toDrive(
    collection=buffered_boundary_fc,
    description='VM0047_v9_Buffered_Boundary_15m_Inset',
    folder=BOUNDARY_EXPORT_FOLDER,
    fileNamePrefix='VM0047_v9_Buffered_Boundary_15m_Inset',
    fileFormat='SHP'
)
start_and_track(t_buffer_boundary, 'VM0047_v9_Buffered_Boundary_15m_Inset (all 127 plots)')
print(f'  Queued buffered-boundary shapefile export (Drive folder "{BOUNDARY_EXPORT_FOLDER}") '
      f'-- independent of RUN_FALLBACK_CARVEOUT, since this is boundary cleanup, not forest carve-out.')

# ------------------------------------------------------------------
# 3. HEIGHT LAYER (Meta/WRI, static/current -- structural evidence)
# ------------------------------------------------------------------
meta_height_raw = (ee.ImageCollection('projects/sat-io/open-datasets/facebook/meta-canopy-height')
                    .filterBounds(study_area)
                    .mosaic()
                    .clip(study_area)
                    .rename('height_raw'))

# ------------------------------------------------------------------
# 4. GEDI CROSS-VALIDATION / HEIGHT BIAS CORRECTION
# ------------------------------------------------------------------
gedi = (ee.ImageCollection('LARSE/GEDI/GEDI02_A_002_MONTHLY')
        .filterBounds(study_area)
        .select(['rh98', 'quality_flag', 'degrade_flag', 'sensitivity']))

def quality_filter_gedi(img):
    good = img.select('quality_flag').eq(1) \
        .And(img.select('degrade_flag').eq(0)) \
        .And(img.select('sensitivity').gt(0.9))
    return img.select('rh98').updateMask(good)

gedi_rh98 = gedi.map(quality_filter_gedi).mosaic().clip(study_area)
gedi_points = gedi_rh98.rename('gedi_h').addBands(meta_height_raw).sample(
    region=study_area, scale=25, numPixels=5000, seed=42, geometries=False
)
diff_fc = gedi_points.map(lambda f: f.set('diff', ee.Number(f.get('gedi_h')).subtract(f.get('height_raw'))))
gedi_sample_count = diff_fc.size().getInfo()
diff_stats = diff_fc.aggregate_stats('diff').getInfo() if gedi_sample_count > 10 else None

MAX_PLAUSIBLE_BIAS_M = 3.0
if diff_stats and diff_stats.get('median') is not None and abs(diff_stats['median']) <= MAX_PLAUSIBLE_BIAS_M:
    HEIGHT_BIAS_CORRECTION_M = diff_stats['median']
    print(f'[GEDI QA] Applied height bias correction (median, m): {HEIGHT_BIAS_CORRECTION_M:.3f}')
else:
    HEIGHT_BIAS_CORRECTION_M = 0.0
    print('[GEDI QA] No correction applied (insufficient samples or bias exceeds sanity cap).')

meta_height = meta_height_raw.add(HEIGHT_BIAS_CORRECTION_M).rename('height')

# ------------------------------------------------------------------
# 5. MULTI-SEASON NDVI COMPOSITES, 2013-2021 HLS ONLY
# ------------------------------------------------------------------
HLS_L30_ID = 'NASA/HLS/HLSL30/v002'
HLS_S30_ID = 'NASA/HLS/HLSS30/v002'

def mask_hls(image):
    qa = image.select('Fmask').toInt()
    cloud, adj_cloud, cloud_shadow, snow = 1 << 1, 1 << 2, 1 << 3, 1 << 4
    clear = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(adj_cloud).eq(0)) \
        .And(qa.bitwiseAnd(cloud_shadow).eq(0)).And(qa.bitwiseAnd(snow).eq(0))
    return image.updateMask(clear)

def hls_l30_ndvi(image):
    img = mask_hls(image).divide(10000)
    return img.normalizedDifference(['B5', 'B4']).rename('NDVI').copyProperties(image, ['system:time_start'])

def hls_s30_ndvi(image):
    img = mask_hls(image).divide(10000)
    return img.normalizedDifference(['B8A', 'B4']).rename('NDVI').copyProperties(image, ['system:time_start'])

def season_composite(year, season_key, region):
    md_start, md_end = SEASON_WINDOWS[season_key]
    start, end = f'{year}-{md_start}', f'{year}-{md_end}'
    l30 = (ee.ImageCollection(HLS_L30_ID).filterDate(start, end).filterBounds(region)
           .filter(ee.Filter.lt('CLOUD_COVERAGE', MAX_CLOUD_COVER_PCT)).map(hls_l30_ndvi))
    s30 = (ee.ImageCollection(HLS_S30_ID).filterDate(start, end).filterBounds(region)
           .filter(ee.Filter.lt('CLOUD_COVERAGE', MAX_CLOUD_COVER_PCT)).map(hls_s30_ndvi))
    merged = l30.merge(s30)
    n_images = merged.size()
    composite = merged.select('NDVI').median()
    count_band = merged.select('NDVI').count().rename('n_images')
    return composite.addBands(count_band).clip(region).set('year', year, 'season', season_key, 'n_images', n_images)

print(f'\nBuilding multi-season HLS composites, 2013-2021 (cloud cover < {MAX_CLOUD_COVER_PCT}%) ...')
season_composites = {}
skipped = []
for y in HLS_YEARS:
    seasons_this_year = SEASONS_PRE_START_2021 if y == PROJECT_START_YEAR else list(SEASON_WINDOWS.keys())
    for s in seasons_this_year:
        img = season_composite(y, s, study_area)
        n = img.get('n_images').getInfo()
        if n == 0:
            skipped.append((y, s))
            print(f'  [{y} / {s}] 0 scene(s) -- SKIPPED (no usable data for this season-year)')
            continue
        season_composites[(y, s)] = img
        flag = ' (thin: <3 scenes)' if n < 3 else ''
        print(f'  [{y} / {s}] {n} scene(s){flag}')

if skipped:
    print(f'\n[Data availability] {len(skipped)} season-year slot(s) skipped for lack of data: {skipped}')
    thin_years = sorted(set(y for y, s in skipped))
    for y in thin_years:
        n_available = sum(1 for (yy, ss) in season_composites if yy == y)
        if n_available < 2:
            print(f'  [WARNING] {y}: only {n_available}/4 season(s) with data -- treat this year\'s '
                  f'contribution to the baseline with reduced confidence.')

baseline_keys = list(season_composites.keys())
baseline_ndvi_collection = ee.ImageCollection(
    [season_composites[k].select('NDVI').set('year', k[0], 'season', k[1]) for k in baseline_keys]
)

def years_with_season(season_key):
    return [y for y in HLS_YEARS if (y, season_key) in season_composites]

# ------------------------------------------------------------------
# 6. MULTI-TEMPORAL EVIDENCE LAYERS
# ------------------------------------------------------------------
veg_binary_collection = baseline_ndvi_collection.map(lambda img: img.gt(NDVI_VEG_THRESHOLD))
persistent_veg_frac = veg_binary_collection.reduce(ee.Reducer.mean()).rename('persistent_veg_frac')

dry_keys = [k for k in baseline_keys if k[1] in DRY_SEASONS]
dry_ndvi_collection = ee.ImageCollection(
    [season_composites[k].select('NDVI').set('year', k[0], 'season', k[1]) for k in dry_keys]
)
dry_n_obs = dry_ndvi_collection.reduce(ee.Reducer.count()).rename('dry_n_obs')
dry_tree_veg_binary = dry_ndvi_collection.map(lambda img: img.gt(NDVI_TREE_THRESHOLD))
persistent_tree_veg_frac = dry_tree_veg_binary.reduce(ee.Reducer.mean()).rename('persistent_tree_veg_frac')

print(f'\n[Baseline persistence] General (all-season) obs pool: {len(baseline_keys)} season-years. '
      f'Dry-season-only (tree) obs pool: {len(dry_keys)} season-years.')

# v10.2: RECENT-ONLY dry-season tree persistence, for the recency-drop
# diagnostic (Section 1d / Section 9) -- same tree-threshold logic as
# persistent_tree_veg_frac above, restricted to RECENT_YEARS_FOR_CLEARING_CHECK
# so a plot that was forested historically but cleared shortly before
# project start reads LOW here even while persistent_tree_veg_frac (all
# years pooled) reads high.
_recent_dry_keys = [k for k in dry_keys if k[0] in RECENT_YEARS_FOR_CLEARING_CHECK]
if _recent_dry_keys:
    recent_dry_ndvi_collection = ee.ImageCollection(
        [season_composites[k].select('NDVI').set('year', k[0], 'season', k[1]) for k in _recent_dry_keys]
    )
    recent_dry_n_obs = recent_dry_ndvi_collection.reduce(ee.Reducer.count()).rename('recent_dry_n_obs')
    recent_dry_tree_veg_binary = recent_dry_ndvi_collection.map(lambda img: img.gt(NDVI_TREE_THRESHOLD))
    recent_persistent_tree_veg_frac = recent_dry_tree_veg_binary.reduce(ee.Reducer.mean()).rename(
        'recent_persistent_tree_veg_frac')
    print(f'  Recency-check obs pool (years {RECENT_YEARS_FOR_CLEARING_CHECK}): '
          f'{len(_recent_dry_keys)} season-years.')
else:
    recent_persistent_tree_veg_frac = None
    print(f'  [WARNING] No dry-season composites available for recency-check years '
          f'{RECENT_YEARS_FOR_CLEARING_CHECK} -- recency-drop columns will be blank for every plot.')

season_climatology = {}
for s in SEASON_WINDOWS:
    yrs = years_with_season(s)
    if yrs:
        season_climatology[s] = ee.ImageCollection([season_composites[(y, s)].select('NDVI') for y in yrs]).mean()
    else:
        print(f'  [WARNING] No surviving composites for season "{s}" across any year -- '
              f'excluded from the seasonal-amplitude calculation.')
season_climatology_stack = ee.ImageCollection(list(season_climatology.values()))
ndvi_seasonal_amplitude = season_climatology_stack.max().subtract(
    season_climatology_stack.min()).rename('ndvi_seasonal_amplitude')

def year_mean_ndvi(year):
    seasons_this_year = SEASONS_PRE_START_2021 if year == PROJECT_START_YEAR else list(SEASON_WINDOWS.keys())
    available = [s for s in seasons_this_year if (year, s) in season_composites]
    if not available:
        return None
    imgs = [season_composites[(year, s)].select('NDVI') for s in available]
    return ee.ImageCollection(imgs).mean().set('year', year)

annual_images = [img for img in (year_mean_ndvi(y) for y in HLS_YEARS) if img is not None]
annual_mean_ndvi = ee.ImageCollection(annual_images)
def add_year_band(img):
    return img.addBands(ee.Image.constant(ee.Number(img.get('year'))).toFloat().rename('year')).select(['year', 'NDVI'])
ndvi_trend_slope = annual_mean_ndvi.map(add_year_band).reduce(ee.Reducer.linearFit()).select('scale').rename('ndvi_trend_slope')

# ------------------------------------------------------------------
# 7. GLCM TEXTURE (supporting evidence only)
# ------------------------------------------------------------------
if (PROJECT_START_YEAR, 'pre_monsoon_dry') not in season_composites:
    raise RuntimeError(
        f'No usable {PROJECT_START_YEAR} pre_monsoon_dry composite survived the '
        f'<{MAX_CLOUD_COVER_PCT}% cloud filter -- this season anchors the current-epoch '
        f'texture layer and cannot be skipped. Raise MAX_CLOUD_COVER_PCT or widen the '
        f'season window and re-run.'
    )
current_ndvi = season_composites[(PROJECT_START_YEAR, 'pre_monsoon_dry')].select('NDVI')
ndvi_8bit = current_ndvi.unitScale(-0.2, 0.9).multiply(255).toByte().rename('ndvi_8bit')
glcm = ndvi_8bit.glcmTexture(size=3)
texture_contrast = glcm.select('ndvi_8bit_contrast').rename('tex_contrast')
texture_entropy = glcm.select('ndvi_8bit_ent').rename('tex_entropy')
texture_homogeneity = glcm.select('ndvi_8bit_idm').rename('tex_homogeneity')

# ------------------------------------------------------------------
# 8. FOUR-CLASS LAND-COVER CLASSIFICATION (1 m grid)
# ------------------------------------------------------------------
persistent_veg_frac_1m = persistent_veg_frac.resample('bilinear').reproject(crs=meta_height.projection(), scale=HEIGHT_SCALE_M)
persistent_tree_veg_frac_1m = persistent_tree_veg_frac.resample('bilinear').reproject(crs=meta_height.projection(), scale=HEIGHT_SCALE_M)
ndvi_seasonal_amplitude_1m = ndvi_seasonal_amplitude.resample('bilinear').reproject(crs=meta_height.projection(), scale=HEIGHT_SCALE_M)
dry_n_obs_1m = dry_n_obs.resample('bilinear').reproject(crs=meta_height.projection(), scale=HEIGHT_SCALE_M)
if recent_persistent_tree_veg_frac is not None:
    recent_persistent_tree_veg_frac_1m = recent_persistent_tree_veg_frac.resample('bilinear').reproject(
        crs=meta_height.projection(), scale=HEIGHT_SCALE_M)
else:
    recent_persistent_tree_veg_frac_1m = None

height_band = meta_height

# ------------------------------------------------------------------
# 8a. DIAGNOSTIC: check why tree_mask might be zeroing out everywhere
#     (temporary -- run once, read the printed stats, then you can
#     comment this block back out; it costs one extra reduceRegion call
#     over the whole study area, which can be slow on large areas)
# ------------------------------------------------------------------
print('\n[Diagnostic] Checking tree_mask input bands over study_area ...')
_diag = ee.Image.cat([
    height_band.rename('height'),
    persistent_tree_veg_frac_1m.rename('tree_persist_frac'),
    ndvi_seasonal_amplitude_1m.rename('amplitude'),
    dry_n_obs_1m.rename('n_obs'),
]).reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.percentile([50, 90]), sharedInputs=True),
    geometry=study_area.geometry(), scale=10, maxPixels=1e10, bestEffort=True
).getInfo()
print(f'  [Whole-study-area, ALL pixels] {_diag}')

# v10: the whole-area numbers above mix scrub/crop/herb pixels in with any real
# tree pixels, which is exactly the wrong population to calibrate a tree/scrub
# discriminator against -- scrub and crop are EXPECTED to show high seasonal
# amplitude and low dry-season persistence; that's the whole point of using
# those two bands to separate them from trees. What actually matters for
# choosing AMPLITUDE_LOW_THRESHOLD and PERSISTENCE_FRACTION is the distribution
# of those same two bands restricted to pixels that already clear the height
# bar (>=2 m) -- i.e. candidates that COULD be trees on structural grounds.
_tall_mask = height_band.gte(MIN_CANOPY_HEIGHT_M)
_diag_tall = ee.Image.cat([
    persistent_tree_veg_frac_1m.rename('tree_persist_frac'),
    ndvi_seasonal_amplitude_1m.rename('amplitude'),
    dry_n_obs_1m.rename('n_obs'),
]).updateMask(_tall_mask).reduceRegion(
    reducer=ee.Reducer.count().combine(
        ee.Reducer.mean().combine(ee.Reducer.percentile([10, 50, 90]), sharedInputs=True),
        sharedInputs=True),
    geometry=study_area.geometry(), scale=10, maxPixels=1e10, bestEffort=True
).getInfo()
print(f'  [Restricted to height >= {MIN_CANOPY_HEIGHT_M} m pixels ONLY] {_diag_tall}')
print(f'  Thresholds in use: MIN_CANOPY_HEIGHT_M={MIN_CANOPY_HEIGHT_M}, '
      f'PERSISTENCE_FRACTION={PERSISTENCE_FRACTION}, '
      f'AMPLITUDE_LOW_THRESHOLD={AMPLITUDE_LOW_THRESHOLD}, '
      f'MIN_OBS_FOR_PERSISTENCE={MIN_OBS_FOR_PERSISTENCE}')
print('  Compare each "_p90" value above against its threshold: whichever band\'s p90 '
      'sits on the wrong side of its threshold is the one zeroing out tree_mask everywhere. '
      'CAVEAT for amplitude specifically: tree_mask requires amplitude < threshold (low '
      'amplitude = candidate tree), so the RELEVANT statistic for that band is p10, not p90 -- '
      'if p10 is already above AMPLITUDE_LOW_THRESHOLD, essentially no pixels pass on '
      'amplitude alone; p90 being above threshold is expected and not itself a problem.')
print('  READ THE "height >= 2m" LINE, NOT THE WHOLE-STUDY-AREA LINE, WHEN CHOOSING '
      'AMPLITUDE_LOW_THRESHOLD OR PERSISTENCE_FRACTION -- if amplitude_p90 among tall '
      'pixels alone is still well above AMPLITUDE_LOW_THRESHOLD, or tree_persist_frac_p90 '
      'among tall pixels is still well below PERSISTENCE_FRACTION, tree_mask will still '
      'zero out regardless of what the whole-area numbers show, and the current threshold '
      'values need another look -- don\'t just widen them further without re-checking this '
      'line after each change.')

herb_mask = height_band.lt(HEIGHT_SCRUB_MIN_M).And(persistent_veg_frac_1m.gte(HERB_MIN_PERSISTENCE))

scrub_mask = height_band.gte(HEIGHT_SCRUB_MIN_M).And(
    height_band.lt(MIN_CANOPY_HEIGHT_M).Or(ndvi_seasonal_amplitude_1m.gte(AMPLITUDE_LOW_THRESHOLD))
)

tree_mask = height_band.gte(MIN_CANOPY_HEIGHT_M) \
    .And(persistent_tree_veg_frac_1m.gte(PERSISTENCE_FRACTION)) \
    .And(ndvi_seasonal_amplitude_1m.lt(AMPLITUDE_LOW_THRESHOLD)) \
    .And(dry_n_obs_1m.gte(MIN_OBS_FOR_PERSISTENCE))

lc_class = ee.Image(0).rename('lc_class').byte()
lc_class = lc_class.where(herb_mask, 1)
lc_class = lc_class.where(scrub_mask, 2)
lc_class = lc_class.where(tree_mask, 3)

# ------------------------------------------------------------------
# 8b. IMMEDIATE TREE_MASK COVERAGE CHECK -- NEW v10
#
# The Section 8a marginals (amplitude and persistence percentiles reported
# SEPARATELY) can't tell you the JOINT pass rate, since tree_mask requires
# both conditions at once -- if the two bands are only weakly correlated,
# a threshold pair that looks individually reasonable can still produce a
# near-empty (or, less obviously, an implausibly large) tree_mask. This
# runs immediately after classification and BEFORE the slow per-parcel
# loop (Section 9) or the subdistrict benchmark (Section 9c), so a bad
# threshold pair is caught in seconds rather than after a full run.
# ------------------------------------------------------------------
print('\n[Diagnostic 8b] Direct tree_mask coverage check (joint condition, not marginals) ...')
_coverage_stats = ee.Image.cat([
    ee.Image(1).rename('all_pixels'),
    _tall_mask.rename('tall_pixels'),
    tree_mask.rename('tree_pixels'),
    scrub_mask.rename('scrub_pixels'),
]).multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=study_area.geometry(), scale=10,
    maxPixels=1e10, bestEffort=True
).getInfo()

_all_ha = (_coverage_stats.get('all_pixels') or 0.0) / 10000.0
_tall_ha = (_coverage_stats.get('tall_pixels') or 0.0) / 10000.0
_tree_ha = (_coverage_stats.get('tree_pixels') or 0.0) / 10000.0
_scrub_ha = (_coverage_stats.get('scrub_pixels') or 0.0) / 10000.0

_tree_pct_of_all = (_tree_ha / _all_ha * 100.0) if _all_ha > 0 else 0.0
_tree_pct_of_tall = (_tree_ha / _tall_ha * 100.0) if _tall_ha > 0 else 0.0

print(f'  Study area total       : {_all_ha:12,.1f} ha')
print(f'  height >= {MIN_CANOPY_HEIGHT_M} m (tall) : {_tall_ha:12,.1f} ha ({_tall_ha/_all_ha*100.0:5.1f}% of study area)' if _all_ha > 0 else '')
print(f'  Tree_Cover (lc_class=3): {_tree_ha:12,.1f} ha ({_tree_pct_of_all:5.1f}% of study area, '
      f'{_tree_pct_of_tall:5.1f}% of the tall-pixel subset)')
print(f'  Scrub_Shrub (lc_class=2): {_scrub_ha:12,.1f} ha ({_scrub_ha/_all_ha*100.0:5.1f}% of study area)' if _all_ha > 0 else '')

if _tree_pct_of_all < 0.5:
    print('  [WARNING] Tree class is still under 0.5% of the whole study area -- tree_mask is '
        'effectively still zeroing out despite the relaxed thresholds. Do not proceed to the '
        'per-parcel loop or trust any PASS result until this is resolved (revisit whether '
        'amplitude and persistence are actually correlated the way assumed, or whether a '
        'third band -- e.g. NDVI_TREE_THRESHOLD itself -- is now the binding constraint).')
elif _tree_pct_of_tall > 60.0:
    print('  [WARNING] Tree class now covers the majority of ALL height>=2m pixels -- given that '
        'Kudal/Sawantwadi are not majority-forested landscapes, this suggests the relaxed '
        'thresholds are over-admitting scrub/tall-crop pixels into the Tree class rather than '
        'isolating genuine canopy. Cross-check against Section 9c (ESA WorldCover / Hansen GFC) '
        'before trusting plot-level results.')
else:
    print(f'  Looks plausible: Tree class is a modest, non-zero share of the tall-pixel subset '
          f'({_tree_pct_of_tall:.1f}%), consistent with real but not majority forest cover. '
          f'Still confirm with Section 9c\'s independent benchmark before final use.')

PIXEL_AREA_M2 = ee.Image.pixelArea()

# ------------------------------------------------------------------
# 9. PER-PARCEL ZONAL STATISTICS, CLASS-AREA BREAKDOWN, AND
#    TRUE-AREA PATCH CONTIGUITY
#
#    NOTE: this loop still runs in full -- Section 9b (subdistrict rollup)
#    and Section 14b (subdistrict raster exports) both depend on its
#    results (df rows + qualifying_polys_by_parcel). Only the *outputs*
#    derived from it (per-parcel CSVs/Excel) are turned off below.
# ------------------------------------------------------------------
plot_list = plots.toList(plots.size())
n_plots = plot_list.size().getInfo()
print(f'\nProcessing {n_plots} parcels: classification + area-based contiguity + trend + texture ...')
print(f'  Primary disqualifying patch-area threshold: {PRIMARY_PATCH_AREA_HA} ha '
      f'(sensitivity sweep at {FOREST_PATCH_AREA_THRESHOLDS_HA} ha)')

rows = []
qualifying_polys_by_parcel = {}

for i in range(n_plots):
    feature = ee.Feature(plot_list.get(i))
    props_in = feature.toDictionary().getInfo()
    raw_geom = feature.geometry()
    name_val = props_in.get(NAME_FIELD, f'index_{i}')

    # v9: analysis runs on the -15 m buffered geometry from Section 2c
    # (falls back to the raw geometry only if that plot's name somehow
    # didn't make it into the Section 2c dict -- shouldn't happen, but
    # fail safe rather than crash the whole loop on one parcel).
    geom = buffered_geom_by_name.get(name_val, raw_geom)
    used_centroid_fallback = buffer_fallback_by_name.get(name_val, False)

    try:
        raw_area_m2 = raw_geom.area(maxError=1).getInfo()
        raw_area_ha = raw_area_m2 / 10000.0
        total_area_m2 = geom.area(maxError=1).getInfo()
        total_area_ha = total_area_m2 / 10000.0

        max_h = meta_height.reduceRegion(
            reducer=ee.Reducer.max(), geometry=geom, scale=HEIGHT_SCALE_M, maxPixels=1e9
        ).get('height').getInfo()

        class_groups = PIXEL_AREA_M2.addBands(lc_class).reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
            geometry=geom, scale=HEIGHT_SCALE_M, maxPixels=1e9
        ).get('groups').getInfo()
        class_area_ha = {k: 0.0 for k in LC_CLASSES}
        for g in class_groups:
            class_area_ha[int(g['class'])] = g['sum'] / 10000.0

        tree_vectors = tree_mask.clip(geom).selfMask().reduceToVectors(
            geometry=geom, scale=HEIGHT_SCALE_M, geometryType='polygon',
            eightConnected=True, maxPixels=1e9
        )
        tree_vectors = tree_vectors.map(lambda f: f.set('area_ha', f.geometry().area(maxError=1).divide(10000)))
        patch_areas_list = tree_vectors.aggregate_array('area_ha').getInfo()
        patch_areas = np.array(patch_areas_list) if patch_areas_list else np.array([])

        tex_stats = ee.Image.cat([texture_contrast, texture_entropy, texture_homogeneity]).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom, scale=HLS_SCALE_M, maxPixels=1e9
        ).getInfo()

        slope = ndvi_trend_slope.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom, scale=HLS_SCALE_M, maxPixels=1e9
        ).get('ndvi_trend_slope').getInfo()

        gedi_shot_count = gedi_rh98.reduceRegion(
            reducer=ee.Reducer.count(), geometry=geom, scale=25, maxPixels=1e8
        ).get('rh98').getInfo() or 0

        # v10.2: recency-drop diagnostic inputs -- historical (all baseline
        # years) vs recent-only (RECENT_YEARS_FOR_CLEARING_CHECK) dry-season
        # tree-persistence fraction, both zonal-meaned over this parcel.
        historical_tree_frac_mean = persistent_tree_veg_frac_1m.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom, scale=HLS_SCALE_M, maxPixels=1e9
        ).get('persistent_tree_veg_frac').getInfo()

        if recent_persistent_tree_veg_frac_1m is not None:
            recent_tree_frac_mean = recent_persistent_tree_veg_frac_1m.reduceRegion(
                reducer=ee.Reducer.mean(), geometry=geom, scale=HLS_SCALE_M, maxPixels=1e9
            ).get('recent_persistent_tree_veg_frac').getInfo()
        else:
            recent_tree_frac_mean = None

        row = dict(props_in)
        row[SUBDIST_FIELD] = NAME_TO_SUBDIST.get(name_val)
        # v9: Total_Parcel_Area_ha / all class-area and pct columns below are
        # computed on the -15 m BUFFERED footprint (this is what was actually
        # sampled). Raw_Parcel_Area_ha is the original digitized plot area,
        # kept for reference so the two can be compared directly.
        row['Total_Parcel_Area_ha'] = total_area_ha
        row['Raw_Parcel_Area_ha'] = raw_area_ha
        row['Buffer_Fallback_Used'] = bool(used_centroid_fallback)
        row['Max_Canopy_Height_m'] = max_h
        for k, label in LC_CLASSES.items():
            ha = class_area_ha[k]
            row[f'Area_{label}_ha'] = ha
            row[f'Pct_{label}'] = (ha / total_area_ha * 100.0) if total_area_ha > 0 else 0.0

        for thr in FOREST_PATCH_AREA_THRESHOLDS_HA:
            cover_ha = float(patch_areas[patch_areas >= thr].sum()) if patch_areas.size else 0.0
            row[f'Tree_Cover_ha_at_{thr}ha_min'] = cover_ha
            row[f'Tree_Cover_pct_at_{thr}ha_min'] = (cover_ha / total_area_ha * 100.0) if total_area_ha > 0 else 0.0

        primary_cover_ha = float(patch_areas[patch_areas >= PRIMARY_PATCH_AREA_HA].sum()) if patch_areas.size else 0.0
        tree_contiguous_pct = (primary_cover_ha / total_area_ha * 100.0) if total_area_ha > 0 else 0.0
        row['Contiguous_Tree_Cover_ha'] = primary_cover_ha
        row['Contiguous_Tree_Cover_pct'] = tree_contiguous_pct
        row['N_Tree_Patches_Total'] = int(patch_areas.size)
        row['N_Tree_Patches_Qualifying'] = int((patch_areas >= PRIMARY_PATCH_AREA_HA).sum()) if patch_areas.size else 0

        row['NDVI_Trend_Slope_per_yr'] = slope
        row['Texture_Contrast'] = tex_stats.get('tex_contrast')
        row['Texture_Entropy'] = tex_stats.get('tex_entropy')
        row['Texture_Homogeneity'] = tex_stats.get('tex_homogeneity')
        row['GEDI_Shots_In_Parcel'] = gedi_shot_count

        # -- v10.2: recency-drop diagnostic columns (evidence only, does
        # not by itself change Final_Result) --
        row['Historical_Dry_Season_Tree_Frac'] = historical_tree_frac_mean
        row['Recent_PreProject_Dry_Season_Tree_Frac'] = recent_tree_frac_mean
        if historical_tree_frac_mean is not None and recent_tree_frac_mean is not None:
            recency_gap = historical_tree_frac_mean - recent_tree_frac_mean
            row['Recency_Drop_in_Tree_Signal'] = recency_gap
            row['Possible_Recent_Clearing_Flag'] = bool(
                recency_gap >= RECENCY_DROP_THRESHOLD and recent_tree_frac_mean <= RECENCY_ABS_LOW_THRESHOLD)
        else:
            row['Recency_Drop_in_Tree_Signal'] = None
            row['Possible_Recent_Clearing_Flag'] = False

        # -- algorithmic (pixel classifier) result -- always computed and
        # always kept, regardless of any manual override below --
        algorithmic_fails = tree_contiguous_pct >= MIN_CONTIGUOUS_COVER_PCT
        row['Final_Result_Algorithmic'] = (
            'Fail - Forest' if algorithmic_fails else f'Pass (<{MIN_CONTIGUOUS_COVER_PCT:.0f}%)')
        # raster/vector evidence of what the classifier itself detected is kept
        # for every plot it flagged, independent of whether a manual override
        # later changes the overall verdict -- a reviewer should be able to see
        # what the pixels showed even for an overridden-to-Pass plot.
        if algorithmic_fails:
            qualifying_polys_by_parcel[name_val] = tree_vectors.filter(ee.Filter.gte('area_ha', PRIMARY_PATCH_AREA_HA))

        # -- v10.2: manual/desk-review override (Section 1c), applied on top
        # of, never instead of, the algorithmic result --
        override = MANUAL_VERIFICATION_OVERRIDES.get(name_val)
        if override is not None:
            override_status = override.get('status')
            if override_status not in ('Confirmed_Non_Forest', 'Confirmed_Forest'):
                print(f'  [WARNING] {name_val}: MANUAL_VERIFICATION_OVERRIDES has an '
                      f'unrecognized status "{override_status}" -- ignoring the override, '
                      f'using the algorithmic result instead.')
                override = None

        if override is not None:
            row['Manual_Override_Applied'] = True
            row['Manual_Override_Status'] = override['status']
            row['Manual_Override_Evidence'] = override.get('evidence', '')
            row['Manual_Override_By'] = override.get('verified_by', '')
            row['Manual_Override_Date'] = override.get('verified_date', '')
            fails = (override['status'] == 'Confirmed_Forest')
        else:
            row['Manual_Override_Applied'] = False
            row['Manual_Override_Status'] = ''
            row['Manual_Override_Evidence'] = ''
            row['Manual_Override_By'] = ''
            row['Manual_Override_Date'] = ''
            fails = algorithmic_fails

        # -- reviewed result: everything downstream (df_pass/df_fail split,
        # Excel sheets, subdistrict rollup) buckets on THIS column --
        row['Final_Result'] = 'Fail - Forest' if fails else f'Pass (<{MIN_CONTIGUOUS_COVER_PCT:.0f}%)'

        if row['Manual_Override_Applied'] and row['Final_Result'] != row['Final_Result_Algorithmic']:
            print(f'  [OVERRIDE] {name_val}: algorithmic result was '
                  f'"{row["Final_Result_Algorithmic"]}" -> reviewed to "{row["Final_Result"]}" '
                  f'per manual verification (Manual_Override_Evidence column has the basis).')

        if fails:
            if slope is not None and slope > TREND_FLAT_BAND:
                row['Flag_Group'] = GROUP_INCREASING
            elif slope is not None and slope < -TREND_FLAT_BAND:
                row['Flag_Group'] = GROUP_DECLINING
            else:
                row['Flag_Group'] = GROUP_STABLE
            row['Evidence_Needed'] = EVIDENCE_NEEDED[row['Flag_Group']]
        else:
            row['Flag_Group'] = ''
            row['Evidence_Needed'] = ''

        recency_note = ' [RECENCY-DROP FLAGGED]' if row['Possible_Recent_Clearing_Flag'] else ''
        console_line = (
            f'[{i+1}/{n_plots}] {name_val}: TreeCover={tree_contiguous_pct:.1f}% '
            f'({row["N_Tree_Patches_Qualifying"]}/{row["N_Tree_Patches_Total"]} patches >= '
            f'{PRIMARY_PATCH_AREA_HA}ha)  (Scrub={row["Pct_Scrub_Shrub"]:.1f}%, '
            f'Herb={row["Pct_Herbaceous_Grass_Crop"]:.1f}%) -> {row["Final_Result"]}'
            f'{" (overridden)" if row["Manual_Override_Applied"] else ""}{recency_note}')
        # v10.3: the exact processing-panel line is stored with the row, so the
        # CSV can be checked against the console output plot by plot.
        row['Console_Line'] = console_line
        row['Processing_Error'] = ''
        row['Run_ID'] = RUN_ID
        rows.append(row)
        print('  ' + console_line)

    except Exception as e:
        print(f'  [{i+1}/{n_plots}] {name_val}: FAILED -- {e}')
        row = dict(props_in)
        row[SUBDIST_FIELD] = NAME_TO_SUBDIST.get(name_val)
        row['Processing_Error'] = str(e)
        # v10.3: previously an errored plot had NO Final_Result, so it fell
        # out of BOTH the Pass and the Flagged files without any trace. It is
        # now labelled explicitly and exported in the Flagged file + register.
        row['Final_Result_Algorithmic'] = 'Error - Not Processed'
        row['Final_Result'] = 'Error - Not Processed'
        row['Manual_Override_Applied'] = False
        row['Flag_Group'] = ''
        row['Evidence_Needed'] = 'Re-run this plot -- processing error, no result computed.'
        row['Console_Line'] = f'[{i+1}/{n_plots}] {name_val}: FAILED -- {e}'
        row['Run_ID'] = RUN_ID
        rows.append(row)

df = pd.DataFrame(rows)

def baseline_status(pct):
    if pd.isna(pct):
        return 'N/A - Processing Error'
    if pct < PURE_FALLOW_MAX_PCT:
        return 'Pure Fallow (<2%)'
    elif pct < MIN_CONTIGUOUS_COVER_PCT:
        return 'Degraded with Scattered Stems'
    return 'Closed / Contiguous Forest'

df['Baseline_Land_Status'] = df['Contiguous_Tree_Cover_pct'].apply(baseline_status)

df_pass = df[df['Final_Result'].astype(str).str.startswith('Pass')].copy()
# v10.3: everything that is NOT a Pass goes to the flagged set (Fail - Forest
# AND processing errors), so no plot can silently vanish from both files.
df_fail = df[~df['Final_Result'].astype(str).str.startswith('Pass')].copy()
assert len(df_pass) + len(df_fail) == len(df), 'Pass + Flagged must cover every plot'

# -- v10.2: confirm every entry in MANUAL_VERIFICATION_OVERRIDES actually
# matched a plot Name -- a silent typo here means the override was never
# applied and the algorithmic result was used with no warning otherwise.
_n_overrides_applied = int(df['Manual_Override_Applied'].sum()) if 'Manual_Override_Applied' in df.columns else 0
print(f'\n[Manual override] {_n_overrides_applied}/{len(MANUAL_VERIFICATION_OVERRIDES)} '
      f'registry entries matched a plot and were applied.')
_matched_names = set(df.loc[df['Manual_Override_Applied'], NAME_FIELD]) if _n_overrides_applied else set()
_unmatched_overrides = set(MANUAL_VERIFICATION_OVERRIDES.keys()) - _matched_names
if _unmatched_overrides:
    print(f'  [WARNING] These override entries did NOT match any plot Name -- check spelling/'
          f'underscores against Re127ProjectPlots exactly: {sorted(_unmatched_overrides)}')

# -- v10.2: systematic recency-drop check across all 127 plots, not just the
# ones manually spot-checked -- same signal (forested historically, not
# forested in the last 1-2 pre-project dry seasons) that prompted the
# overrides above, surfaced automatically for every remaining plot.
if 'Possible_Recent_Clearing_Flag' in df.columns:
    _recency_flagged = df[df['Possible_Recent_Clearing_Flag'] == True]  # noqa: E712
    if len(_recency_flagged):
        print(f'\n[Recency check] {len(_recency_flagged)} parcel(s) show a drop of >= '
              f'{RECENCY_DROP_THRESHOLD:.2f} between historical (all baseline years) and recent '
              f'({RECENT_YEARS_FOR_CLEARING_CHECK}) dry-season tree-persistence, with the recent '
              f'value at or below {RECENCY_ABS_LOW_THRESHOLD:.2f}. This is the SAME pattern the 4 '
              f'manually-reviewed plots above showed -- worth the same multi-date Google Earth Pro '
              f'check, even for parcels whose Final_Result already reads Pass:')
        print(_recency_flagged[[NAME_FIELD, 'Final_Result', 'Manual_Override_Applied',
                                 'Historical_Dry_Season_Tree_Frac',
                                 'Recent_PreProject_Dry_Season_Tree_Frac']].to_string(index=False))
    else:
        print('\n[Recency check] No additional parcels show a large historical-vs-recent '
              'persistence drop beyond the ones already manually reviewed.')

print(f'\n=== OVERALL: {len(df_pass)} parcels PASS ({df_pass["Total_Parcel_Area_ha"].sum():.2f} ha), '
      f'{len(df_fail)} parcels FLAGGED ({df_fail["Total_Parcel_Area_ha"].sum():.2f} ha) ===')
if len(df_fail):
    print(df_fail.groupby('Flag_Group')['Total_Parcel_Area_ha'].agg(['count', 'sum']).to_string())
    print('\nClass-area totals within flagged parcels (ha):')
    print(df_fail[[f'Area_{v}_ha' for v in LC_CLASSES.values()]].sum().to_string())
    print('\n[Primary defense output] Every flagged parcel now has a Flag_Group + '
          'Evidence_Needed entry for manual high-resolution/revenue-record defense '
          'review. Fallback carve-out is '
          f'{"ENABLED" if RUN_FALLBACK_CARVEOUT else "DISABLED"} (RUN_FALLBACK_CARVEOUT) -- '
          'see Section 10.')

# ------------------------------------------------------------------
# 9a. MANUAL VERIFICATION REGISTER + CHECKPOINT EXPORT -- NEW v10.3
#
# The register lists every plot a human needs to look at:
#   - every plot the PIXEL CLASSIFIER failed (Final_Result_Algorithmic =
#     'Fail - Forest'), whether or not it has been overridden yet
#   - every plot with a processing error
#   - every plot the recency-drop diagnostic flagged, even if it passed
# Review_Status says exactly where each one stands:
#   'REVIEWED - Confirmed Non-Forest (override applied)'
#   'REVIEWED - Confirmed Forest'
#   'PENDING - manual verification required (classifier: Fail - Forest)'
#   'PENDING - re-run required (processing error)'
#   'RECOMMENDED - recency-drop flag (classifier: Pass)'
# To clear a PENDING row, do the multi-date Google Earth Pro review and add
# the plot to MANUAL_VERIFICATION_OVERRIDES (Section 1c), then re-run.
# ------------------------------------------------------------------
REGISTER_COLS = [
    NAME_FIELD, SUBDIST_FIELD, 'Raw_Parcel_Area_ha', 'Total_Parcel_Area_ha',
    'Contiguous_Tree_Cover_pct', 'N_Tree_Patches_Qualifying', 'N_Tree_Patches_Total',
    'Pct_Scrub_Shrub', 'Pct_Herbaceous_Grass_Crop', 'Max_Canopy_Height_m',
    'NDVI_Trend_Slope_per_yr', 'Historical_Dry_Season_Tree_Frac',
    'Recent_PreProject_Dry_Season_Tree_Frac', 'Recency_Drop_in_Tree_Signal',
    'Possible_Recent_Clearing_Flag',
    'Final_Result_Algorithmic', 'Final_Result', 'Review_Reason', 'Review_Status',
    'Manual_Override_Status', 'Manual_Override_Evidence', 'Manual_Override_By',
    'Manual_Override_Date', 'Flag_Group', 'Evidence_Needed', 'Processing_Error',
    'Console_Line', 'Run_ID',
]


def build_verification_register(frame):
    algo = frame['Final_Result_Algorithmic'].astype(str)
    overridden = frame['Manual_Override_Applied'].fillna(False).astype(bool)
    recency = (frame['Possible_Recent_Clearing_Flag'].fillna(False).astype(bool)
               if 'Possible_Recent_Clearing_Flag' in frame.columns
               else pd.Series(False, index=frame.index))
    algo_fail = algo.str.startswith('Fail')
    is_error = algo.str.startswith('Error')
    needs = algo_fail | is_error | recency | overridden
    reg = frame[needs].copy()

    def reason(r):
        parts = []
        a = str(r.get('Final_Result_Algorithmic', ''))
        if a.startswith('Fail'):
            parts.append('Classifier result: Fail - Forest')
        if a.startswith('Error'):
            parts.append('Processing error')
        if bool(r.get('Possible_Recent_Clearing_Flag', False)):
            parts.append('Recency-drop flag')
        if bool(r.get('Manual_Override_Applied', False)) and not a.startswith('Fail'):
            parts.append('Manual override on a classifier Pass')
        return '; '.join(parts)

    def status(r):
        a = str(r.get('Final_Result_Algorithmic', ''))
        if a.startswith('Error'):
            return 'PENDING - re-run required (processing error)'
        if bool(r.get('Manual_Override_Applied', False)):
            if r.get('Manual_Override_Status') == 'Confirmed_Non_Forest':
                return 'REVIEWED - Confirmed Non-Forest (override applied)'
            return 'REVIEWED - Confirmed Forest'
        if a.startswith('Fail'):
            return 'PENDING - manual verification required (classifier: Fail - Forest)'
        return 'RECOMMENDED - recency-drop flag (classifier: Pass)'

    if len(reg):
        reg['Review_Reason'] = reg.apply(reason, axis=1)
        reg['Review_Status'] = reg.apply(status, axis=1)
        # pending first, then reviewed, then recommended
        order = {'PENDING': 0, 'REVIEWED': 1, 'RECOMMENDED': 2}
        reg['_o'] = reg['Review_Status'].str.split(' ').str[0].map(order)
        reg = reg.sort_values(['_o', NAME_FIELD]).drop(columns='_o')
    else:
        reg['Review_Reason'] = []
        reg['Review_Status'] = []
    return reg[[c for c in REGISTER_COLS if c in reg.columns]]


verification_register_df = build_verification_register(df)

print('\n=== MANUAL VERIFICATION REGISTER (v10.3) ===')
if len(verification_register_df):
    print(verification_register_df[[NAME_FIELD, 'Contiguous_Tree_Cover_pct',
                                    'Final_Result_Algorithmic', 'Final_Result',
                                    'Review_Status']].to_string(index=False))
    _n_pending = int(verification_register_df['Review_Status'].str.startswith('PENDING').sum())
    _n_algo_fail = int(df['Final_Result_Algorithmic'].astype(str).str.startswith('Fail').sum())
    print(f'\n  Classifier Fail - Forest: {_n_algo_fail} plot(s) | '
          f'overridden to Pass after review: {int((df["Manual_Override_Applied"].fillna(False).astype(bool) & df["Final_Result"].astype(str).str.startswith("Pass")).sum())} | '
          f'final Fail - Forest: {int(df["Final_Result"].astype(str).str.startswith("Fail").sum())} | '
          f'PENDING manual verification: {_n_pending}')
else:
    print('  No plots need manual verification.')


def write_parcel_outputs(stage_label):
    """Write the per-parcel CSVs from the in-memory results, then re-read
    them and check they match. Called right after Section 9 (checkpoint,
    so a later slow/failed step cannot leave you with stale files) and
    again at Section 11."""
    files = {
        'VM0047_v9_All_Parcels_Detail.csv': df,
        'VM0047_v9_Fully_Passing_Parcels.csv': df_pass,
        'VM0047_v9_Flagged_Parcels.csv': df_fail,
        'VM0047_v9_Manual_Verification_Register.csv': verification_register_df,
    }
    print(f'\n[{stage_label}] Writing per-parcel CSVs to {OUTPUT_DIR}')
    for fname, frame in files.items():
        frame.to_csv(out_path(fname), index=False)
        print(f'  [CSV] {out_path(fname)}  ({len(frame)} rows)')

    # -- reconciliation: re-read from disk and compare with memory --
    problems = []
    for fname, frame in files.items():
        disk = pd.read_csv(out_path(fname))
        if len(disk) != len(frame):
            problems.append(f'{fname}: {len(disk)} rows on disk vs {len(frame)} in memory')
        elif len(frame) and set(disk[NAME_FIELD].astype(str)) != set(frame[NAME_FIELD].astype(str)):
            problems.append(f'{fname}: plot names on disk differ from memory')
        if len(disk) and 'Run_ID' in disk.columns and 'Run_ID' in frame.columns:
            if set(disk['Run_ID'].astype(str)) != {RUN_ID}:
                problems.append(f'{fname}: contains rows from another run')
    passing_on_disk = set(pd.read_csv(out_path('VM0047_v9_Fully_Passing_Parcels.csv'))[NAME_FIELD].astype(str)) \
        if len(df_pass) else set()
    failing_now = set(df.loc[df['Final_Result'].astype(str).str.startswith('Fail'), NAME_FIELD].astype(str))
    leaked = passing_on_disk & failing_now
    if leaked:
        problems.append(f'Fail - Forest plot(s) found in the Fully_Passing CSV: {sorted(leaked)}')

    if problems:
        print(f'  [RECONCILE] *** MISMATCH -- do NOT use these files ***')
        for p in problems:
            print(f'    - {p}')
    else:
        print(f'  [RECONCILE] OK -- files match the processing panel: '
              f'{len(df_pass)} Pass, {int(df["Final_Result"].astype(str).str.startswith("Fail").sum())} '
              f'Fail - Forest, {int(df["Final_Result"].astype(str).str.startswith("Error").sum())} error(s), '
              f'{len(verification_register_df)} in the verification register.')


write_parcel_outputs('Checkpoint after Section 9')

# ------------------------------------------------------------------
# 9b. SUBDISTRICT-LEVEL RESULTS AGGREGATION
# ------------------------------------------------------------------
print('\nAggregating results by subdistrict ...')

if SUBDIST_FIELD not in df.columns:
    print(f'  [WARNING] "{SUBDIST_FIELD}" not found on parcel attributes -- '
          f'subdistrict summary will be empty.')
    subdist_summary_df = pd.DataFrame()
else:
    class_area_cols = [f'Area_{v}_ha' for v in LC_CLASSES.values()]

    def summarize_subdist(sub_name, frame):
        d = {SUBDIST_FIELD: sub_name}
        d['N_Parcels_Total'] = len(frame)
        d['N_Parcels_Pass'] = int(frame['Final_Result'].astype(str).str.startswith('Pass').sum())
        d['N_Parcels_Flagged'] = int(frame['Final_Result'].astype(str).str.startswith('Fail').sum())
        d['Total_Parcel_Area_ha'] = frame['Total_Parcel_Area_ha'].sum()
        d['Passing_Area_ha'] = frame.loc[frame['Final_Result'].astype(str).str.startswith('Pass'),
                                          'Total_Parcel_Area_ha'].sum()
        d['Flagged_Area_ha'] = frame.loc[frame['Final_Result'].astype(str).str.startswith('Fail'),
                                          'Total_Parcel_Area_ha'].sum()
        for c in class_area_cols:
            if c in frame.columns:
                d[c] = frame[c].sum()
        d['Mean_Contiguous_Tree_Cover_pct'] = frame['Contiguous_Tree_Cover_pct'].mean()
        for grp in (GROUP_STABLE, GROUP_DECLINING, GROUP_INCREASING):
            d[f'N_Flagged_{grp}'] = int((frame['Flag_Group'] == grp).sum())
        return d

    subdist_rows = [summarize_subdist(sub, df[df[SUBDIST_FIELD] == sub])
                     for sub in STUDY_SUBDISTRICTS if (df[SUBDIST_FIELD] == sub).any()]

    other_mask = ~df[SUBDIST_FIELD].isin(STUDY_SUBDISTRICTS)
    if other_mask.any():
        for sub_name, frame in df[other_mask].groupby(SUBDIST_FIELD):
            subdist_rows.append(summarize_subdist(sub_name, frame))
            print(f'  [NOTE] Parcel(s) with "{SUBDIST_FIELD}" = "{sub_name}" are outside '
                  f'STUDY_SUBDISTRICTS {STUDY_SUBDISTRICTS} -- included anyway, add them to '
                  f'the list above if this is expected on every run.')

    subdist_summary_df = pd.DataFrame(subdist_rows)
    print(subdist_summary_df.to_string(index=False))

# ------------------------------------------------------------------
# 9c. SUBDISTRICT-WIDE FOREST VALIDATION (FSI BENCHMARK CHECK) -- NEW v8
#
# WHY: Section 9b only sums up the 127 plots, so its numbers can never
# disagree with the per-plot results -- it is not an independent check.
# This section runs the SAME tree classification over the FULL Kudal /
# Sawantwadi admin polygons from study_area. Both subdistricts contain
# real, established forest tracts, so a healthy pipeline MUST report
# substantial forest here. Three numbers per subdistrict:
#
#   (1) Pipeline forest %  -- this script's tree_mask, sieved at the
#       DNA 0.05 ha minimum patch, evaluated on a 10 m grid
#   (2) ESA WorldCover 2021 tree-cover %   (independent 10 m product)
#   (3) Hansen GFC forest % as of end-2020 (treecover2000 >= 15%,
#       minus mapped loss through 2020)
#
# INTERPRETATION (printed automatically):
#   - (1) roughly in line with (2)/(3)  -> pipeline registers true forest;
#     the plot-level pass results are credible and PRR-defensible.
#   - (1) near zero while (2)/(3) show real forest -> tree_mask is
#     zeroing out (thresholds too strict / dry_n_obs too low -- see the
#     Section 8a diagnostic) and NO plot result should be trusted yet.
#
# Also compare (1) against the official FSI ISFR figure by hand: paste
# the published percentage into FSI_REFERENCE_FOREST_PCT below and the
# delta is printed for the audit record.
# ------------------------------------------------------------------
print('\n=== 9c. SUBDISTRICT-WIDE FOREST VALIDATION (FSI benchmark) ===')

VALIDATION_SCALE_M = 10
DNA_MIN_PATCH_HA = 0.05
MIN_PATCH_PIXELS_10M = int(round(DNA_MIN_PATCH_HA * 10000.0 / (VALIDATION_SCALE_M ** 2)))  # 5 px

# Fill these in from the FSI India State of Forest Report (ISFR) -- use the
# subdistrict figure if published, else the closest district figure with a
# note. Leave None to skip the FSI delta line for that subdistrict.
FSI_REFERENCE_FOREST_PCT = {
    'Kudal': None,
    'Sawantwadi': None,
}

# Aggregate the 1 m tree_mask to a 10 m majority grid, then sieve out
# connected patches smaller than the DNA minimum mappable area (0.05 ha).
#
# FIX (v10.1): tree_mask is built from boolean .And() combinations of
# several reprojected/resampled bands, and that chain of ops loses GEE's
# notion of a fixed "default projection" (the pyramid/scale info
# reduceResolution needs to know what it's aggregating FROM). Without it,
# reduceResolution throws "does not have a valid default projection" --
# stamping meta_height's 1 m projection back onto tree_mask before the
# call tells GEE explicitly "treat this as a 1 m grid in this CRS", which
# is true (it IS built on the 1 m Meta/WRI height grid), so this is a
# metadata fix, not a resampling change.
_meta_height_source_proj = ( ee.ImageCollection('projects/sat-io/open-datasets/facebook/meta-canopy-height').filterBounds(study_area) .first() .projection() )
projection_10m = _meta_height_source_proj.atScale(VALIDATION_SCALE_M)
tree_mask_1m_proj = tree_mask.setDefaultProjection(_meta_height_source_proj)
tree_fraction_10m = (tree_mask_1m_proj.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024) .reproject(projection_10m))
# projection_10m = meta_height.projection().atScale(VALIDATION_SCALE_M)
# tree_mask_1m_proj = tree_mask.setDefaultProjection(meta_height.projection())
# tree_fraction_10m = (tree_mask_1m_proj
#                      .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
#                      .reproject(projection_10m))
tree_binary_10m = tree_fraction_10m.gte(0.5)
patch_pixel_count_10m = tree_binary_10m.selfMask().connectedPixelCount(
    maxSize=1024, eightConnected=True)
tree_sieved_10m = tree_binary_10m.selfMask().updateMask(
    patch_pixel_count_10m.gte(MIN_PATCH_PIXELS_10M))

# Independent benchmark layers
worldcover_map = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
worldcover_tree_mask = worldcover_map.eq(10)   # WorldCover class 10 = Tree cover

hansen_gfc = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
hansen_loss_year = hansen_gfc.select('lossyear').unmask(0)   # 0 = no loss; 1..23 = 2001..2023
hansen_forest_2020_mask = (hansen_gfc.select('treecover2000').gte(15)
                           .And(hansen_loss_year.eq(0).Or(hansen_loss_year.gt(20))))

def masked_area_ha(mask_image, region_geometry, scale_m):
    area_m2 = (ee.Image.pixelArea().updateMask(mask_image).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region_geometry,
        scale=scale_m, maxPixels=1e11, bestEffort=True
    ).get('area').getInfo())
    return (area_m2 or 0.0) / 10000.0

validation_rows = []
for subdistrict_name in STUDY_SUBDISTRICTS:
    # v10.3: a timeout/error on one subdistrict no longer kills the run
    # (parcel CSVs were already written at the Section 9a checkpoint).
    try:
        boundary_fc = study_area.filter(ee.Filter.eq(SUBDIST_FIELD, subdistrict_name))
        if boundary_fc.size().getInfo() == 0:
            print(f'  [{subdistrict_name}] no study_area polygon with {SUBDIST_FIELD} = '
                  f'"{subdistrict_name}" -- validation skipped for it')
            continue

        boundary_geometry = boundary_fc.geometry()
        subdistrict_area_ha = boundary_geometry.area(maxError=1).getInfo() / 10000.0
        print(f'  [{subdistrict_name}] computing forest areas over the full admin polygon '
              f'({subdistrict_area_ha:,.0f} ha) -- this is the slow part, please wait ...')

        pipeline_forest_ha = masked_area_ha(tree_sieved_10m, boundary_geometry, VALIDATION_SCALE_M)
        worldcover_forest_ha = masked_area_ha(worldcover_tree_mask, boundary_geometry, VALIDATION_SCALE_M)
        hansen_forest_ha = masked_area_ha(hansen_forest_2020_mask, boundary_geometry, 30)

        pipeline_pct = pipeline_forest_ha / subdistrict_area_ha * 100.0
        worldcover_pct = worldcover_forest_ha / subdistrict_area_ha * 100.0
        hansen_pct = hansen_forest_ha / subdistrict_area_ha * 100.0

        print(f'    Pipeline forest (DNA-sieved) : {pipeline_forest_ha:12,.1f} ha  ({pipeline_pct:5.1f}%)')
        print(f'    ESA WorldCover 2021 tree     : {worldcover_forest_ha:12,.1f} ha  ({worldcover_pct:5.1f}%)')
        print(f'    Hansen GFC forest (end 2020) : {hansen_forest_ha:12,.1f} ha  ({hansen_pct:5.1f}%)')

        fsi_reference_pct = FSI_REFERENCE_FOREST_PCT.get(subdistrict_name)
        if fsi_reference_pct is not None:
            print(f'    FSI ISFR reference           : {fsi_reference_pct:5.1f}%  '
                  f'(delta vs pipeline: {pipeline_pct - fsi_reference_pct:+.1f} pp)')
        else:
            print(f'    FSI ISFR reference           : not entered -- paste it into '
                  f'FSI_REFERENCE_FOREST_PCT for the audit record.')

        benchmark_pct = max(worldcover_pct, hansen_pct)
        if benchmark_pct >= 5.0 and pipeline_pct < 0.25 * benchmark_pct:
            verdict = ('UNDER-DETECTING FOREST: independent products show real forest here '
                       'but the pipeline barely registers any. tree_mask is zeroing out -- '
                       'revisit MIN_CANOPY_HEIGHT_M / PERSISTENCE_FRACTION / '
                       'AMPLITUDE_LOW_THRESHOLD / MIN_OBS_FOR_PERSISTENCE against the '
                       'Section 8a diagnostic BEFORE trusting any plot-level pass.')
        elif benchmark_pct >= 5.0 and pipeline_pct > 2.0 * benchmark_pct:
            verdict = ('OVER-DETECTING FOREST vs benchmarks -- check whether scrub is '
                       'leaking into the Tree class (amplitude threshold) before use.')
        else:
            verdict = ('CONSISTENT with independent benchmarks -- the pipeline registers '
                       'true forest tracts, so plot-level pass results are credible.')
        print(f'    VERDICT: {verdict}')

        validation_rows.append({
            SUBDIST_FIELD: subdistrict_name,
            'Subdistrict_Area_ha': subdistrict_area_ha,
            'Pipeline_Forest_ha': pipeline_forest_ha,
            'Pipeline_Forest_pct': pipeline_pct,
            'WorldCover2021_Tree_ha': worldcover_forest_ha,
            'WorldCover2021_Tree_pct': worldcover_pct,
            'HansenGFC_Forest2020_ha': hansen_forest_ha,
            'HansenGFC_Forest2020_pct': hansen_pct,
            'FSI_ISFR_Reference_pct': fsi_reference_pct,
            'Verdict': verdict.split(':')[0].split(' vs')[0],
        })
    except Exception as _val_err:
        print(f'  [{subdistrict_name}] validation SKIPPED -- {_val_err}. Parcel results '
              f'are unaffected; re-run Section 9c on its own later if needed.')

validation_df = pd.DataFrame(validation_rows)
if len(validation_df):
    validation_csv_path = out_path('VM0047_v9_Subdistrict_Forest_Validation.csv')
    validation_df.to_csv(validation_csv_path, index=False)
    print(f'\n[Validation] Written to {validation_csv_path} (local working directory).')

    # v10: this run changed PERSISTENCE_FRACTION and AMPLITUDE_LOW_THRESHOLD from their
    # previous values. Section 9c's independent ESA WorldCover / Hansen GFC comparison is
    # the objective check on whether that change was a fix or an overcorrection -- print
    # an explicit gate on it rather than letting the "CONSISTENT" verdict scroll by unread.
    print('\n=== v10 THRESHOLD-CHANGE QUALITY GATE ===')
    print(f'  New values this run: PERSISTENCE_FRACTION={PERSISTENCE_FRACTION}, '
          f'AMPLITUDE_LOW_THRESHOLD={AMPLITUDE_LOW_THRESHOLD}')
    for _, vr in validation_df.iterrows():
        v = vr['Verdict']
        sub = vr[SUBDIST_FIELD]
        if 'CONSISTENT' in v:
            print(f'  [{sub}] OK -- pipeline forest % now lines up with ESA WorldCover / Hansen GFC.')
        elif 'OVER-DETECTING' in v:
            print(f'  [{sub}] WARNING -- pipeline now reports MORE forest than independent products. '
                  f'The relaxed thresholds may be admitting scrub/crop into the Tree class. Do not '
                  f'treat plot-level PASS results from this run as final until this is resolved.')
        elif 'UNDER-DETECTING' in v:
            print(f'  [{sub}] WARNING -- pipeline still reports far less forest than independent '
                  f'products despite the relaxed thresholds. Something else in the classification '
                  f'chain is still zeroing out tree_mask -- re-check the height>=2m diagnostic line '
                  f'in Section 8a before trusting any plot-level result.')
        else:
            print(f'  [{sub}] {v} -- see full verdict text printed above for detail.')

# ------------------------------------------------------------------
# 10. SUB-PARCEL CARVE-OUT -- GATED FALLBACK, NOT AUTOMATIC
#     (unchanged -- off by default via RUN_FALLBACK_CARVEOUT = False;
#     qualifying_polys_by_parcel is still needed by Section 14b)
# ------------------------------------------------------------------
carve_df = pd.DataFrame()
if RUN_FALLBACK_CARVEOUT:
    print('\n[Fallback carve-out ENABLED] Carving eligible (non-Tree-Cover) sub-polygons '
          'out of flagged parcels ...')
    carve_rows = []
    for _, r in df_fail.iterrows():
        name_val = r.get(NAME_FIELD)
        if name_val not in qualifying_polys_by_parcel:
            continue
        match = plots.filter(ee.Filter.eq(NAME_FIELD, name_val)).first()
        geom = ee.Feature(match).geometry()
        qualifying_polys = qualifying_polys_by_parcel[name_val]

        try:
            n_forest_polys = qualifying_polys.size().getInfo()
            eligible_geom = geom.difference(qualifying_polys.geometry().dissolve(maxError=1), maxError=1) \
                if n_forest_polys > 0 else geom

            eligible_area_ha = ee.Geometry(eligible_geom).area(maxError=1).getInfo() / 10000.0
            forest_area_ha = r['Total_Parcel_Area_ha'] - eligible_area_ha

            if eligible_area_ha >= CARVE_MIN_REMAINDER_HA:
                carve_rows.append({
                    NAME_FIELD: name_val, SUBDIST_FIELD: r.get(SUBDIST_FIELD),
                    'Flag_Group': r['Flag_Group'],
                    'Original_Parcel_Area_ha': r['Total_Parcel_Area_ha'],
                    'Area_Scrub_Shrub_ha': r['Area_Scrub_Shrub_ha'],
                    'Area_Herbaceous_Grass_Crop_ha': r['Area_Herbaceous_Grass_Crop_ha'],
                    'Excluded_Tree_Cover_Area_ha': forest_area_ha,
                    'Carved_Eligible_Area_ha': eligible_area_ha,
                })
                print(f'  {name_val}: carved {eligible_area_ha:.2f} ha eligible '
                      f'(excluded {forest_area_ha:.2f} ha as qualifying Tree Cover, '
                      f'>= {PRIMARY_PATCH_AREA_HA} ha per patch)')
            else:
                print(f'  {name_val}: carved remainder ({eligible_area_ha:.2f} ha) below '
                      f'{CARVE_MIN_REMAINDER_HA} ha minimum -- parcel excluded entirely')
        except Exception as e:
            print(f'  {name_val}: carve-out FAILED -- {e}')

    carve_df = pd.DataFrame(carve_rows)
else:
    print('\n[Fallback carve-out DISABLED] RUN_FALLBACK_CARVEOUT is False -- no sub-parcel '
          'geometry has been cut.')

# ------------------------------------------------------------------
# 9c. OVERALL ELIGIBILITY SUMMARY (restored -- needed by CSV/Excel below)
# ------------------------------------------------------------------
original_total_ha = df['Total_Parcel_Area_ha'].sum()
pass_total_ha = df_pass['Total_Parcel_Area_ha'].sum()
carved_total_ha = carve_df['Carved_Eligible_Area_ha'].sum() if len(carve_df) else None
excluded_tree_ha = carve_df['Excluded_Tree_Cover_Area_ha'].sum() if len(carve_df) else None
revised_eligible_total_ha = (pass_total_ha + carved_total_ha) if carved_total_ha is not None else None

summary_df = pd.DataFrame([{
    'Run_ID': RUN_ID,
    'Total_Parcels_Processed': len(df),
    'Classifier_Fail_Forest_Count': int(df['Final_Result_Algorithmic'].astype(str).str.startswith('Fail').sum()),
    'Overridden_To_Pass_After_Manual_Review': int((df['Manual_Override_Applied'].fillna(False).astype(bool)
                                                   & df['Final_Result'].astype(str).str.startswith('Pass')
                                                   & df['Final_Result_Algorithmic'].astype(str).str.startswith('Fail')).sum()),
    'Final_Fail_Forest_Count': int(df['Final_Result'].astype(str).str.startswith('Fail').sum()),
    'Processing_Error_Count': int(df['Final_Result'].astype(str).str.startswith('Error').sum()),
    'Pending_Manual_Verification_Count': int(verification_register_df['Review_Status'].astype(str).str.startswith('PENDING').sum()) if len(verification_register_df) else 0,
    'Original_Total_Parcel_Area_ha': original_total_ha,
    'Fully_Passing_Area_ha': pass_total_ha,
    'Fully_Passing_Parcel_Count': len(df_pass),
    'Flagged_Parcel_Count': len(df_fail),
    'Flagged_Parcel_Area_ha': df_fail['Total_Parcel_Area_ha'].sum() if len(df_fail) else 0.0,
    'Total_Scrub_Shrub_Area_ha_in_Flagged': df_fail['Area_Scrub_Shrub_ha'].sum() if len(df_fail) else 0.0,
    'Total_Herbaceous_Area_ha_in_Flagged': df_fail['Area_Herbaceous_Grass_Crop_ha'].sum() if len(df_fail) else 0.0,
    'Total_Tree_Cover_Area_ha_in_Flagged': df_fail['Area_Tree_Cover_ha'].sum() if len(df_fail) else 0.0,
    'Fallback_Carveout_Run': RUN_FALLBACK_CARVEOUT,
    'Carved_Eligible_Area_ha': carved_total_ha if carved_total_ha is not None else 'N/A - fallback not run',
    'Excluded_Tree_Cover_Area_ha': excluded_tree_ha if excluded_tree_ha is not None else 'N/A - fallback not run',
    'Revised_Total_Eligible_Area_ha': revised_eligible_total_ha if revised_eligible_total_ha is not None else 'N/A - fallback not run (see Fully_Passing_Area_ha as the current defensible floor)',
}])
print('\n=== ELIGIBILITY SUMMARY ===')
print(summary_df.to_string(index=False))

# ------------------------------------------------------------------
# 11. CSV EXPORTS  -- ALL RESTORED
#
# NOTE ON "I wasn't getting any CSVs/Excel": these .to_csv()/wb.save()
# calls write to the CURRENT WORKING DIRECTORY of the Python process
# running this script -- NOT to Google Drive. If you ran this in
# Colab, that's Colab's ephemeral /content/ (or wherever you `cd`'d
# to) -- it disappears when the runtime resets and is separate from
# your Drive. If you ran it locally, it's whatever folder you launched
# the script from. Only the GEE raster exports (Section 14/14b) go to
# Drive, and only once poll_tasks() reports them COMPLETED. The print
# below shows the exact absolute path each file was written to so you
# can go find it.
# ------------------------------------------------------------------
import os  # defensive re-import: safe even if already imported above --
           # protects against NameError if you're running this cell-by-cell
           # in a notebook and skipped re-running the top-of-file imports.
_cwd = os.getcwd()
print(f'\n[CSV/Excel] Writing outputs to this run\'s folder: {OUTPUT_DIR}')

# v10.3: per-parcel CSVs (incl. the Manual Verification Register) are written
# and reconciled by the same helper used at the Section 9a checkpoint.
write_parcel_outputs('Section 11 final export')

if len(carve_df):
    carve_df.to_csv(out_path('VM0047_v9_Carved_Eligible_Subpolygons.csv'), index=False)
    print(f'[CSV] {out_path("VM0047_v9_Carved_Eligible_Subpolygons.csv")}')

summary_df.to_csv(out_path('VM0047_v9_Eligibility_Summary.csv'), index=False)
print(f'[CSV] {out_path("VM0047_v9_Eligibility_Summary.csv")}')

if len(subdist_summary_df):
    subdist_summary_df.to_csv(out_path('VM0047_v9_Subdistrict_Summary.csv'), index=False)
    print(f'[CSV] {out_path("VM0047_v9_Subdistrict_Summary.csv")}')
else:
    print('[CSV] subdist_summary_df is empty -- Subdistrict_Summary.csv not written. Check '
          f'SUBDIST_FIELD ("{SUBDIST_FIELD}") matches your asset\'s attribute name.')

# ------------------------------------------------------------------
# 12. STYLED, MULTI-SHEET EXCEL
#     OUTPUT-SCOPE EDIT: Excel workbook build/write is disabled this run
#     (it was per-parcel-sheet focused). Function definition kept in case
#     you want to re-enable it later -- just uncomment the call at the
#     bottom of this section.
# ------------------------------------------------------------------
def export_styled_excel_v9(df_pass, df_fail, carve_df, summary_df, subdist_summary_df, path):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    HEADER_FILL = PatternFill(start_color='1F4E29', end_color='1F4E29', fill_type='solid')
    HEADER_FONT = Font(color='FFFFFF', bold=True, size=11)
    PASS_FILL = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
    FAIL_FILL = PatternFill(start_color='FCE4E4', end_color='FCE4E4', fill_type='solid')
    ALT_ROW = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
    THIN = Side(style='thin', color='B7B7B7')
    BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

    def write_sheet(ws, frame, col_map):
        cols = [(c, h) for c, h in col_map if c in frame.columns]
        for c_idx, (_, header) in enumerate(cols, start=1):
            cell = ws.cell(row=1, column=c_idx, value=header)
            cell.fill, cell.font = HEADER_FILL, HEADER_FONT
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = BORDER
        for r_idx, (_, row) in enumerate(frame.iterrows(), start=2):
            for c_idx, (col, _) in enumerate(cols, start=1):
                val = row.get(col)
                if isinstance(val, float):
                    val = round(val, 2)
                cell = ws.cell(row=r_idx, column=c_idx, value=val)
                cell.border = BORDER
                cell.alignment = Alignment(horizontal='left', vertical='center')
                if col in ('Final_Result', 'Final_Result_Algorithmic') and isinstance(val, str):
                    cell.fill = PASS_FILL if val.startswith('Pass') else FAIL_FILL
                elif r_idx % 2 == 0:
                    cell.fill = ALT_ROW
        for c_idx, (col, header) in enumerate(cols, start=1):
            ws.column_dimensions[get_column_letter(c_idx)].width = max(len(header) + 2, 14)
        ws.freeze_panes = 'A2'

    wb = Workbook()

    ws1 = wb.active
    ws1.title = 'Fully_Passing_Parcels'
    write_sheet(ws1, df_pass, [
        (NAME_FIELD, 'Name'), (SUBDIST_FIELD, 'Sub-district'),
        ('Raw_Parcel_Area_ha', 'Raw Parcel Area (ha)'),
        ('Total_Parcel_Area_ha', 'Buffered Analysis Area (ha, -15m inset)'),
        ('Buffer_Fallback_Used', 'Centroid Fallback Used?'),
        ('Max_Canopy_Height_m', 'Max Canopy Height (m)'),
        ('Area_Tree_Cover_ha', 'Tree Cover Area (ha)'),
        ('Area_Scrub_Shrub_ha', 'Scrub/Shrub Area (ha)'),
        ('Area_Herbaceous_Grass_Crop_ha', 'Herbaceous/Crop Area (ha)'),
        ('Area_Bare_Fallow_ha', 'Bare/Fallow Area (ha)'),
        ('Contiguous_Tree_Cover_pct', 'Contiguous Tree Cover (%)'),
        ('N_Tree_Patches_Qualifying', f'Patches >= {PRIMARY_PATCH_AREA_HA} ha'),
        ('Final_Result_Algorithmic', 'Pixel Classifier Result (unreviewed)'),
        ('Manual_Override_Applied', 'Manual Override Applied?'),
        ('Manual_Override_Evidence', 'Manual Override Evidence'),
        ('Manual_Override_By', 'Reviewed By'),
        ('Manual_Override_Date', 'Review Date'),
        ('Recency_Drop_in_Tree_Signal', 'Historical-vs-Recent Tree Signal Drop'),
        ('Possible_Recent_Clearing_Flag', 'Recency-Drop Flagged?'),
        ('Final_Result', 'India DNA Forest Threshold (15%) -- Reviewed'),
        ('Baseline_Land_Status', 'Baseline Land Status'),
    ])

    ws2 = wb.create_sheet('Flagged_Parcels_by_Group')
    write_sheet(ws2, df_fail, [
        (NAME_FIELD, 'Name'), (SUBDIST_FIELD, 'Sub-district'),
        ('Raw_Parcel_Area_ha', 'Raw Parcel Area (ha)'),
        ('Total_Parcel_Area_ha', 'Buffered Analysis Area (ha, -15m inset)'),
        ('Buffer_Fallback_Used', 'Centroid Fallback Used?'),
        ('Final_Result_Algorithmic', 'Pixel Classifier Result (unreviewed)'),
        ('Final_Result', 'Final Result (after review)'),
        ('Processing_Error', 'Processing Error'),
        ('Flag_Group', 'Group'),
        ('Area_Tree_Cover_ha', 'Tree Cover Area (ha)'),
        ('Pct_Tree_Cover', 'Tree Cover (%)'),
        ('N_Tree_Patches_Qualifying', f'Patches >= {PRIMARY_PATCH_AREA_HA} ha'),
        ('N_Tree_Patches_Total', 'Total Tree Patches (any size)'),
        ('Area_Scrub_Shrub_ha', 'Scrub/Shrub Area (ha)'),
        ('Pct_Scrub_Shrub', 'Scrub/Shrub (%)'),
        ('Area_Herbaceous_Grass_Crop_ha', 'Herbaceous/Crop Area (ha)'),
        ('Pct_Herbaceous_Grass_Crop', 'Herbaceous/Crop (%)'),
        ('Area_Bare_Fallow_ha', 'Bare/Fallow Area (ha)'),
        ('Contiguous_Tree_Cover_pct', 'Contiguous Tree Cover (%)'),
        ('Tree_Cover_pct_at_0.05ha_min', 'Tree Cover % @0.05ha min'),
        ('Tree_Cover_pct_at_0.25ha_min', 'Tree Cover % @0.25ha min'),
        ('Tree_Cover_pct_at_0.5ha_min', 'Tree Cover % @0.5ha min'),
        ('NDVI_Trend_Slope_per_yr', 'NDVI Trend Slope (/yr)'),
        ('Historical_Dry_Season_Tree_Frac', 'Historical Dry-Season Tree Frac (all years)'),
        ('Recent_PreProject_Dry_Season_Tree_Frac', f'Recent Dry-Season Tree Frac ({RECENT_YEARS_FOR_CLEARING_CHECK})'),
        ('Recency_Drop_in_Tree_Signal', 'Historical-vs-Recent Tree Signal Drop'),
        ('Possible_Recent_Clearing_Flag', 'Recency-Drop Flagged?'),
        ('Manual_Override_Applied', 'Manual Override Applied?'),
        ('Texture_Contrast', 'Texture Contrast'),
        ('Texture_Entropy', 'Texture Entropy'),
        ('Evidence_Needed', 'Evidence Needed (primary defense)'),
    ])

    ws3 = wb.create_sheet('Carved_Eligible_Subpolygons')
    if len(carve_df):
        write_sheet(ws3, carve_df, [
            (NAME_FIELD, 'Name'), (SUBDIST_FIELD, 'Sub-district'),
            ('Flag_Group', 'Group'),
            ('Original_Parcel_Area_ha', 'Original Parcel Area (ha)'),
            ('Area_Scrub_Shrub_ha', 'Scrub/Shrub Area (ha)'),
            ('Area_Herbaceous_Grass_Crop_ha', 'Herbaceous/Crop Area (ha)'),
            ('Excluded_Tree_Cover_Area_ha', 'Excluded Tree Cover Area (ha)'),
            ('Carved_Eligible_Area_ha', 'Carved Eligible Area (ha)'),
        ])
    else:
        msg = ('Fallback carve-out is DISABLED (RUN_FALLBACK_CARVEOUT = False). Complete the '
               'Group-based defense review on the Flagged_Parcels_by_Group sheet first.') \
            if not RUN_FALLBACK_CARVEOUT else 'No sub-polygons carved (see console log).'
        ws3.cell(row=1, column=1, value=msg)

    ws_reg = wb.create_sheet('Manual_Verification_Register', 0)
    if len(verification_register_df):
        write_sheet(ws_reg, verification_register_df, [
            (NAME_FIELD, 'Name'), (SUBDIST_FIELD, 'Sub-district'),
            ('Review_Status', 'Review Status'),
            ('Review_Reason', 'Why It Needs Review'),
            ('Final_Result_Algorithmic', 'Pixel Classifier Result (unreviewed)'),
            ('Final_Result', 'Final Result (after review)'),
            ('Contiguous_Tree_Cover_pct', 'Contiguous Tree Cover (%)'),
            ('N_Tree_Patches_Qualifying', f'Patches >= {PRIMARY_PATCH_AREA_HA} ha'),
            ('N_Tree_Patches_Total', 'Total Tree Patches'),
            ('Pct_Scrub_Shrub', 'Scrub/Shrub (%)'),
            ('Pct_Herbaceous_Grass_Crop', 'Herbaceous/Crop (%)'),
            ('Max_Canopy_Height_m', 'Max Canopy Height (m)'),
            ('Raw_Parcel_Area_ha', 'Raw Parcel Area (ha)'),
            ('Total_Parcel_Area_ha', 'Buffered Analysis Area (ha)'),
            ('Recency_Drop_in_Tree_Signal', 'Historical-vs-Recent Tree Signal Drop'),
            ('Possible_Recent_Clearing_Flag', 'Recency-Drop Flagged?'),
            ('Manual_Override_Status', 'Override Status'),
            ('Manual_Override_Evidence', 'Manual Verification Evidence'),
            ('Manual_Override_By', 'Reviewed By'),
            ('Manual_Override_Date', 'Review Date'),
            ('Evidence_Needed', 'Evidence Needed'),
            ('Processing_Error', 'Processing Error'),
            ('Console_Line', 'Processing Panel Line'),
            ('Run_ID', 'Run ID'),
        ])
        # colour the status cells: red = pending, green = reviewed, amber = recommended
        _status_col = 3
        for _r in range(2, ws_reg.max_row + 1):
            _c = ws_reg.cell(row=_r, column=_status_col)
            _v = str(_c.value or '')
            if _v.startswith('PENDING'):
                _c.fill = FAIL_FILL
                _c.font = Font(bold=True, color='9C0006')
            elif _v.startswith('REVIEWED'):
                _c.fill = PASS_FILL
            else:
                _c.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
    else:
        ws_reg.cell(row=1, column=1, value='No plots need manual verification in this run.')

    ws4 = wb.create_sheet('Eligibility_Summary')
    write_sheet(ws4, summary_df, [(c, c.replace('_', ' ')) for c in summary_df.columns])

    ws5 = wb.create_sheet('Subdistrict_Summary')
    if len(subdist_summary_df):
        write_sheet(ws5, subdist_summary_df, [(c, c.replace('_', ' ')) for c in subdist_summary_df.columns])
    else:
        ws5.cell(row=1, column=1, value=f'No "{SUBDIST_FIELD}" values found on parcel attributes.')

    wb.save(path)
    print(f'[Excel] v9 workbook written to {path}')

_excel_path = out_path('VM0047_v9_Eligibility_Report.xlsx')
export_styled_excel_v9(df_pass, df_fail, carve_df, summary_df, subdist_summary_df,
                        _excel_path)
print(f'[Excel] {_excel_path}')

# ------------------------------------------------------------------
# 12b. DIRECT BROWSER DOWNLOAD (Colab) -- NEW v10.1
#
# The CSVs/Excel above only get written to Colab's ephemeral local disk
# (see the Section 11 note) -- nothing forces them into your browser's
# Downloads folder on its own. This zips every CSV/Excel produced this
# run into one archive and triggers a single browser download via
# google.colab.files.download(). One zip instead of one files.download()
# call per file because Chrome/Firefox silently block a burst of several
# simultaneous downloads unless you've allowed multi-download for this
# site -- a single zip sidesteps that entirely.
# ------------------------------------------------------------------
import zipfile

_possible_outputs = [
    'VM0047_v9_All_Parcels_Detail.csv',
    'VM0047_v9_Fully_Passing_Parcels.csv',
    'VM0047_v9_Flagged_Parcels.csv',
    'VM0047_v9_Manual_Verification_Register.csv',
    'VM0047_v9_Carved_Eligible_Subpolygons.csv',
    'VM0047_v9_Eligibility_Summary.csv',
    'VM0047_v9_Subdistrict_Summary.csv',
    'VM0047_v9_Subdistrict_Forest_Validation.csv',
    'VM0047_v9_Eligibility_Report.xlsx',
]
_existing_outputs = [p for p in _possible_outputs if os.path.exists(out_path(p))]

# v10.3: zip name carries the RUN_ID, so a download can never be confused
# with an older run's bundle sitting in your Downloads folder.
_zip_path = os.path.abspath(f'VM0047_v9_Results_Bundle_{RUN_ID}.zip')
with zipfile.ZipFile(_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in _existing_outputs:
        zf.write(out_path(p), arcname=p)

print(f'\n[Download] Bundled {len(_existing_outputs)} file(s) into {_zip_path}:')
for p in _existing_outputs:
    print(f'    - {p}')

try:
    from google.colab import files as _colab_files
    _colab_files.download(_zip_path)
    print('[Download] Browser download triggered. If nothing appears, check this '
          'tab\'s popup/download-blocker icon in the address bar and allow it, '
          'then re-run just this cell (no need to re-run the whole analysis).')
except ImportError:
    print('[Download] google.colab not available (not running in Colab) -- '
          f'grab {_zip_path} directly from the working directory instead.')
except Exception as e:
    print(f'[Download] files.download() failed ({e}). The zip still exists at '
          f'{_zip_path} -- open the Colab file browser '
          '(folder icon, left sidebar), find it there, and download manually.')

# ------------------------------------------------------------------
# 13. REVISED ELIGIBLE PROJECT BOUNDARY -- GATED, same fallback flag
#     (unchanged -- stays off by default since RUN_FALLBACK_CARVEOUT = False)
# ------------------------------------------------------------------
revised_boundary_fc = None
if RUN_FALLBACK_CARVEOUT and len(carve_df):
    print('\nAssembling revised eligible project boundary for export ...')
    boundary_features = []
    for _, r in df_pass.iterrows():
        match = plots.filter(ee.Filter.eq(NAME_FIELD, r.get(NAME_FIELD))).first()
        geom = ee.Feature(match).geometry()
        boundary_features.append(ee.Feature(geom, {
            NAME_FIELD: r.get(NAME_FIELD), SUBDIST_FIELD: r.get(SUBDIST_FIELD),
            'Source': 'Fully_Passing', 'Eligible_Area_ha': r['Total_Parcel_Area_ha']
        }))
    for _, r in carve_df.iterrows():
        name_val = r[NAME_FIELD]
        if name_val not in qualifying_polys_by_parcel:
            continue
        match = plots.filter(ee.Filter.eq(NAME_FIELD, name_val)).first()
        parcel_geom = ee.Feature(match).geometry()
        qualifying_polys = qualifying_polys_by_parcel[name_val]
        eligible_geom = parcel_geom.difference(qualifying_polys.geometry().dissolve(maxError=1), maxError=1)
        boundary_features.append(ee.Feature(eligible_geom, {
            NAME_FIELD: name_val, SUBDIST_FIELD: r[SUBDIST_FIELD],
            'Source': 'Carved_Eligible', 'Eligible_Area_ha': r['Carved_Eligible_Area_ha']
        }))

    revised_boundary_fc = ee.FeatureCollection(boundary_features)
    t_boundary = ee.batch.Export.table.toDrive(
        collection=revised_boundary_fc,
        description='VM0047_v9_Revised_Eligible_Project_Boundary',
        folder=CARVEOUT_EXPORT_FOLDER,
        fileNamePrefix='VM0047_v9_Revised_Eligible_Project_Boundary',
        fileFormat='SHP'
    )
    start_and_track(t_boundary, 'VM0047_v9_Revised_Eligible_Project_Boundary')
    print(f'  Queued shapefile export (Drive folder "{CARVEOUT_EXPORT_FOLDER}")')
else:
    print('\n[Boundary export SKIPPED] Fallback carve-out disabled -- no revised boundary '
          'shapefile generated this run.')

# ------------------------------------------------------------------
# 14. PER-PLOT RASTER EXPORTS -- RE-ENABLED in v9 (were commented out
#     in v7/v8). Region is the RAW plot geometry + a small outward
#     EXPORT_BUFFER_M margin for visual context around each parcel --
#     that is unrelated to the -15 m inward analysis inset in Section
#     2c; the qualifying-patch band still reflects the buffered-geometry
#     analysis from Section 9.
# ------------------------------------------------------------------
# print(f'\nQueuing per-plot raster exports to Drive folder "{EXPORT_DRIVE_FOLDER}" ...')
# for i in range(n_plots):
#     feature = ee.Feature(plot_list.get(i))
#     props_in = feature.toDictionary().getInfo()
#     name_val = props_in.get(NAME_FIELD, f'plot_{i}')
#     safe = safe_name(name_val)
#     geom_buf = feature.geometry().buffer(EXPORT_BUFFER_M)

#     if name_val in qualifying_polys_by_parcel:
#         qualifying_raster = ee.Image().byte().paint(qualifying_polys_by_parcel[name_val], 1).unmask(0)
#     else:
#         qualifying_raster = ee.Image(0).byte()

#     patch_band_name = f'tree_patch_qualifying_ge_{safe_band_suffix(PRIMARY_PATCH_AREA_HA)}ha'

#     stack_1m = ee.Image.cat([
#         meta_height.rename('height_calibrated_m'),
#         lc_class.rename('lc_class'),
#         qualifying_raster.rename(patch_band_name),
#     ]).clip(geom_buf)
#     t1 = ee.batch.Export.image.toDrive(
#         image=stack_1m, description=f'VM0047_{safe}_1m_classified_stack', folder=EXPORT_DRIVE_FOLDER,
#         fileNamePrefix=f'{safe}_1m_classified_stack', region=geom_buf, scale=HEIGHT_SCALE_M,
#         maxPixels=1e9, fileFormat='GeoTIFF'
#     )
#     start_and_track(t1, f'{safe}_1m_classified_stack')

#     stack_30m = ee.Image.cat([
#         persistent_veg_frac.rename('persistent_veg_frac_all_season').toFloat(),
#         persistent_tree_veg_frac.rename('persistent_tree_veg_frac_dry_season').toFloat(),
#         ndvi_seasonal_amplitude.rename('ndvi_seasonal_amplitude').toFloat(),
#         dry_n_obs.rename('dry_n_obs').toFloat(),
#         ndvi_trend_slope.toFloat(),
#         texture_contrast.toFloat(), texture_entropy.toFloat(), texture_homogeneity.toFloat(),
#     ]).clip(geom_buf)
#     t2 = ee.batch.Export.image.toDrive(
#         image=stack_30m, description=f'VM0047_{safe}_30m_evidence_stack', folder=EXPORT_DRIVE_FOLDER,
#         fileNamePrefix=f'{safe}_30m_evidence_stack', region=geom_buf, scale=HLS_SCALE_M,
#         maxPixels=1e9, fileFormat='GeoTIFF'
#     )
#     start_and_track(t2, f'{safe}_30m_evidence_stack')

#     print(f'  [{i+1}/{n_plots}] {safe}: queued classified 1m + evidence 30m export tasks')

# ------------------------------------------------------------------
# 14b. SUBDISTRICT-LEVEL RASTER EXPORTS -- KEPT ACTIVE
#      (this is the raster output the run should still produce)
# ------------------------------------------------------------------
# print(f'\nQueuing subdistrict-level raster exports to Drive folder '
#       f'"{SUBDISTRICT_EXPORT_FOLDER}" ...')

# patch_band_name = f'tree_patch_qualifying_ge_{safe_band_suffix(PRIMARY_PATCH_AREA_HA)}ha'

# for sub in STUDY_SUBDISTRICTS:
#     # Plots don't carry SUBDIST_FIELD server-side (only study_area does) --
#     # use the client-side NAME_TO_SUBDIST mapping from Section 2b to get the
#     # list of plot Names in this subdistrict, then filter plots by Name
#     # (an 'in' filter works server-side since NAME_FIELD does exist on plots).
#     names_in_sub = [nm for nm, s in NAME_TO_SUBDIST.items() if s == sub]
#     if not names_in_sub:
#         print(f'  [{sub}] no parcels matched to subdistrict "{sub}" via the spatial join -- skipped')
#         continue
#     sub_plots = plots.filter(ee.Filter.inList(NAME_FIELD, names_in_sub))
#     n_sub_plots = sub_plots.size().getInfo()
#     if n_sub_plots == 0:
#         print(f'  [{sub}] no parcels found with {SUBDIST_FIELD} = "{sub}" -- skipped')
#         continue

#     safe_sub = safe_name(sub)
#     sub_geom = sub_plots.geometry().dissolve(maxError=1).buffer(EXPORT_BUFFER_M)

#     sub_names_in_this_subdist = [
#         r.get(NAME_FIELD) for r in
#         df[df[SUBDIST_FIELD] == sub].to_dict('records')
#     ] if SUBDIST_FIELD in df.columns else []
#     qualifying_fcs_this_sub = [
#         qualifying_polys_by_parcel[nm] for nm in sub_names_in_this_subdist
#         if nm in qualifying_polys_by_parcel
#     ]

#     if qualifying_fcs_this_sub:
#         merged_qualifying_fc = qualifying_fcs_this_sub[0]
#         for fc in qualifying_fcs_this_sub[1:]:
#             merged_qualifying_fc = merged_qualifying_fc.merge(fc)
#         sub_qualifying_raster = ee.Image().byte().paint(merged_qualifying_fc, 1).unmask(0)
#     else:
#         sub_qualifying_raster = ee.Image(0).byte()

#     stack_1m_sub = ee.Image.cat([
#         meta_height.rename('height_calibrated_m'),
#         lc_class.rename('lc_class'),
#         sub_qualifying_raster.rename(patch_band_name),
#     ]).clip(sub_geom)
#     t1_sub = ee.batch.Export.image.toDrive(
#         image=stack_1m_sub, description=f'VM0047_{safe_sub}_1m_classified_stack',
#         folder=SUBDISTRICT_EXPORT_FOLDER, fileNamePrefix=f'{safe_sub}_1m_classified_stack',
#         region=sub_geom, scale=HEIGHT_SCALE_M, maxPixels=1e10, fileFormat='GeoTIFF'
#     )
#     start_and_track(t1_sub, f'{safe_sub}_1m_classified_stack (subdistrict)')

#     stack_30m_sub = ee.Image.cat([
#         persistent_veg_frac.rename('persistent_veg_frac_all_season').toFloat(),
#         persistent_tree_veg_frac.rename('persistent_tree_veg_frac_dry_season').toFloat(),
#         ndvi_seasonal_amplitude.rename('ndvi_seasonal_amplitude').toFloat(),
#         dry_n_obs.rename('dry_n_obs').toFloat(),
#         ndvi_trend_slope.toFloat(),
#         texture_contrast.toFloat(), texture_entropy.toFloat(), texture_homogeneity.toFloat(),
#     ]).clip(sub_geom)
#     t2_sub = ee.batch.Export.image.toDrive(
#         image=stack_30m_sub, description=f'VM0047_{safe_sub}_30m_evidence_stack',
#         folder=SUBDISTRICT_EXPORT_FOLDER, fileNamePrefix=f'{safe_sub}_30m_evidence_stack',
#         region=sub_geom, scale=HLS_SCALE_M, maxPixels=1e10, fileFormat='GeoTIFF'
#     )
#     start_and_track(t2_sub, f'{safe_sub}_30m_evidence_stack (subdistrict)')

#     print(f'  [{sub}] {n_sub_plots} parcel(s) dissolved into one boundary -- queued '
#           f'classified 1m + evidence 30m export tasks')

# print('\n[Monitor] Polling all queued export tasks for completion/failure ...')
# poll_tasks(_all_tasks, poll_interval_sec=30)

# ------------------------------------------------------------------
# 15. NOT AUTOMATABLE IN GEE -- recommended manual/parallel evidence
# ------------------------------------------------------------------
print('\n[Recommendation] For the primary defense of flagged parcels (before any '
      'fallback carve-out), supplement the class-area breakdown and Flag_Group with: '
      '(a) Google Earth historical high-resolution imagery snapshots across the '
      'baseline years, (b) revenue/land-use records (7/12 extracts or equivalent) '
      'confirming agricultural or fallow classification, and (c) site photographs. '
      'This run did not produce the per-parcel Excel sheet that normally holds the '
      'Flag_Group / Evidence_Needed columns -- re-enable Section 12 if you need it '
      'for that review.')

# ------------------------------------------------------------------
# 16. OPTIONAL: QA MAP (requires geemap) -- unchanged
# ------------------------------------------------------------------
try:
    import geemap.foliumap as geemap
    Map = geemap.Map()
    Map.centerObject(study_area, 12)
    Map.addLayer(lc_class, {'min': 0, 'max': 3,
                 'palette': ['d9c89e', 'c2e07a', '4f9d3d', '1b5e20']},
                 'Land Cover Classification (Bare/Herb/Scrub/Tree)')
    Map.addLayer(tree_mask, {'palette': ['1b5e20']}, 'Tree Cover class (all patch sizes, raw)')
    Map.addLayer(persistent_tree_veg_frac, {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'darkgreen']},
                 'Dry-Season Tree Persistence Fraction', False)
    Map.addLayer(ndvi_seasonal_amplitude, {'min': 0, 'max': 0.5, 'palette': ['blue', 'white', 'red']},
                 'NDVI Seasonal Amplitude (scrub/crop discriminator)', False)
    Map.addLayer(meta_height, {'min': 0, 'max': 25, 'palette': ['white', 'green', 'darkgreen']}, 'Meta/WRI Height', False)
    if revised_boundary_fc is not None:
        Map.addLayer(revised_boundary_fc, {}, 'Revised Eligible Project Boundary (fallback)')
    Map.addLayer(plots, {}, 'Original Stakeholder Parcels', True)
    Map.addLayerControl()
except Exception as e:
    Map = None
    print(f'\n[QA Map SKIPPED] geemap display failed ({e}). This does not affect the '
          f'CSV results or the raster exports queued/polled above -- it only '
          f'means the inline preview map did not render in this run.')

Map

[Run] RUN_ID = 20260923_083935
[Run] All CSV/Excel outputs for THIS run go to: /content/VM0047_outputs_20260923_083935

[Field check] Properties found on plots (projects/ee-saleemkhan/assets/Re127ProjectPlots):
  ['Name', 'OID_', 'Shape_Area', 'Shape_Leng', 'system:index']
[Field check] Properties found on study_area (projects/ee-saleemkhan/assets/EarthMAA_StudyArea):
  ['Dist_LGD', 'District', 'OBJECTID', 'REMARKS', 'STATE_LGD', 'STATE_UT', 'Shape_Area', 'Shape_Leng', 'Sub_dist', 'Subdis_LGD', 'Subdis_Typ', 'system:index']

Spatially joining plots to study_area to assign subdistrict ...
  Matched 123/123 plots to a study_area subdistrict polygon.

Applying -15 m inward buffer in EPSG:32643 to all 127 plots ...
  [8/127] 109_Sudhakar Sawant 5: -15 m inset collapsed the polygon (raw area 1.520 ha) -- FELL BACK to a 2.0 m centroid disc (0.0009 ha). Flag this plot for manual boundary review.
  [42/127] 178_Dasharath_3: -15 m inset collapsed the polygon (raw area 1.149 ha) -- FELL BACK to 

  [22/127] 127_Narayan Kadam_5: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=5.1%, Herb=65.6%) -> Pass (<15%)
  [23/127] 133_Shaligram_all: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=4.3%, Herb=31.8%) -> Pass (<15%)
  [24/127] 138_Sahadev_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=9.9%, Herb=66.6%) -> Pass (<15%)
  [25/127] 139_Prakash_3-4: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=4.6%, Herb=25.0%) -> Pass (<15%)
  [26/127] 149_Krishna_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.3%, Herb=0.0%) -> Pass (<15%)
  [27/127] 155_Divakar: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=9.1%, Herb=21.3%) -> Pass (<15%)
  [28/127] 157_Harishchandra 1 4 5: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=1.6%, Herb=0.2%) -> Pass (<15%)
  [29/127] 157_Harishchandra 13: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=17.0%, Herb=33.1%) -> Pass (<15%)
  [30/127] 157_Harishchandra_9_12_: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=9.7%, Herb=44.6%) -> Pass (<15%)
  [31/127]

  [60/127] 218_Guruprakash_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=6.6%, Herb=28.1%) -> Pass (<15%)
  [61/127] 220_232_Uday_Vishwanath_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.4%, Herb=99.0%) -> Pass (<15%)
  [62/127] 221_Prakash_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=10.5%, Herb=21.8%) -> Pass (<15%)
  [63/127] 225_Santosh_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=2.7%, Herb=85.6%) -> Pass (<15%)
  [64/127] 226_Sachin_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=1.5%, Herb=92.9%) -> Pass (<15%)
  [65/127] 230_1+231_3: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.0%, Herb=0.0%) -> Pass (<15%)
  [66/127] 233_Ramchandra_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=7.5%, Herb=11.0%) -> Pass (<15%)
  [67/127] 233_Ramchandra_9: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=2.2%, Herb=90.1%) -> Pass (<15%)
  [68/127] 233_Ramchandra_10: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.8%, Herb=98.0%) -> Pass (<15%)
  [69/127] 233_Ramchand

  [99/127] 277_Jayprakash_Kanak: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=1.0%, Herb=1.1%) -> Pass (<15%)
  [100/127] 277_Jayprakash_Kanak_3: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=7.8%, Herb=10.5%) -> Pass (<15%)
  [101/127] 277_Jayprakash_Kanak_4: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=16.6%, Herb=51.0%) -> Pass (<15%)
  [102/127] 279_Somaji_Vitthal_Kanak: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=4.3%, Herb=4.8%) -> Pass (<15%)
  [103/127] 280_Subhash_Ramchandra: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=9.0%, Herb=62.8%) -> Pass (<15%)
  [104/127] 282_Vaibhavi_1: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.3%, Herb=0.0%) -> Pass (<15%)
  [105/127] 282_Vinod_2: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.1%, Herb=0.0%) -> Pass (<15%)
  [106/127] 284_Narayan: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=3.0%, Herb=1.9%) -> Pass (<15%)
  [107/127] 285_3_BAMBOO: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=0.0%, Herb=0.0%) -> Pass (<15%)
  

  [OVERRIDE] 315_Praveen Desai_1: algorithmic result was "Fail - Forest" -> reviewed to "Pass (<15%)" per manual verification (Manual_Override_Evidence column has the basis).
  [123/127] 315_Praveen Desai_1: TreeCover=18.1% (1/7 patches >= 0.05ha)  (Scrub=4.5%, Herb=75.1%) -> Pass (<15%) (overridden)
  [124/127] 315_Praveen Desai_2: TreeCover=0.0% (0/3 patches >= 0.05ha)  (Scrub=4.2%, Herb=94.5%) -> Pass (<15%)
  [125/127] 316_Madhukar Desai: TreeCover=0.0% (0/4 patches >= 0.05ha)  (Scrub=12.0%, Herb=83.9%) -> Pass (<15%)


  [126/127] 317_Uttam Nhanu: TreeCover=10.3% (2/34 patches >= 0.05ha)  (Scrub=12.6%, Herb=58.1%) -> Pass (<15%)
  [127/127] 318_Harish Chandra: TreeCover=0.0% (0/0 patches >= 0.05ha)  (Scrub=10.3%, Herb=87.2%) -> Pass (<15%)

[Manual override] 4/4 registry entries matched a plot and were applied.

[Recency check] No additional parcels show a large historical-vs-recent persistence drop beyond the ones already manually reviewed.

=== OVERALL: 123 parcels PASS (356.32 ha), 4 parcels FLAGGED (8.71 ha) ===
            count       sum
Flag_Group                 
Declining       1  1.188112
Increasing      1  0.776372
Stable          2  6.748675

Class-area totals within flagged parcels (ha):
Area_Bare_Fallow_ha              0.000000
Area_Herbaceous_Grass_Crop_ha    1.489254
Area_Scrub_Shrub_ha              0.791224
Area_Tree_Cover_ha               6.402163

[Primary defense output] Every flagged parcel now has a Flag_Group + Evidence_Needed entry for manual high-resolution/revenue-record def

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[Download] Browser download triggered. If nothing appears, check this tab's popup/download-blocker icon in the address bar and allow it, then re-run just this cell (no need to re-run the whole analysis).

[Boundary export SKIPPED] Fallback carve-out disabled -- no revised boundary shapefile generated this run.

[Recommendation] For the primary defense of flagged parcels (before any fallback carve-out), supplement the class-area breakdown and Flag_Group with: (a) Google Earth historical high-resolution imagery snapshots across the baseline years, (b) revenue/land-use records (7/12 extracts or equivalent) confirming agricultural or fallow classification, and (c) site photographs. This run did not produce the per-parcel Excel sheet that normally holds the Flag_Group / Evidence_Needed columns -- re-enable Section 12 if you need it for that review.

[QA Map SKIPPED] geemap display failed ("'Box' object has no attribute 'xyz_to_folium'"). This does not affect the CSV results or the raster ex